---
# **Portfolio Optimization under Markov-Modulated Shifted Wishart Process**

---

# Preliminaries

## Libraries

In [ ]:
import numpy as np

# Example matrices to test
Sigma_inf_normal_T = np.array([[0.0219, 0.0235], [0.0235, 0.0745]])  # Wishart matrix for normal period (T)
Sigma_inf_total_T = np.array([[0.0370, 0.0400], [0.0400, 0.1107]])   # Wishart matrix for total data (T)
Sigma_inf_normal_N = np.array([[0.0150, 0.0176], [0.0176, 0.0653]])  # Wishart matrix for normal period (N)
Sigma_inf_total_N = np.array([[0.0370, 0.0400], [0.0400, 0.1107]])   # Wishart matrix for total data (N)

# Function to compute and print eigenvalues of a covariance matrix
def print_eigenvalues(matrix, label):
    eigenvalues, _ = np.linalg.eig(matrix)
    print(f"Eigenvalues of {label} matrix: {eigenvalues}")

# Printing eigenvalues for Sigma_inf matrices
#print_eigenvalues(Sigma_inf_normal_T, "Sigma_inf_normal_T")
#print_eigenvalues(Sigma_inf_total_T, "Sigma_inf_total_T")
#print_eigenvalues(Sigma_inf_normal_N, "Sigma_inf_normal_N")
#print_eigenvalues(Sigma_inf_total_N, "Sigma_inf_total_N")

print_eigenvalues(Sigma_inf_B, "Sigma_inf_B")
print_eigenvalues(Sigma_inf_M, "Sigma_inf_M")
print_eigenvalues(Sigma_inf_W, "Sigma_inf_W")
print_eigenvalues(Sigma_inf_SW, "Sigma_inf_SW")
print_eigenvalues(Sigma_inf_MW, "Sigma_inf_MW")

# If other Sigma_inf matrices (e.g., for crisis periods) are defined, include them similarly:
# print_eigenvalues(Sigma_inf_crisis, "Sigma_inf_crisis")

Eigenvalues of Sigma_inf_B matrix: [0.00379777 0.04420223]
Eigenvalues of Sigma_inf_M matrix: [0.01946321 0.12823679]
Eigenvalues of Sigma_inf_W matrix: [0.00379777 0.04420223]
Eigenvalues of Sigma_inf_SW matrix: [0.01946321 0.12823679]
Eigenvalues of Sigma_inf_MW matrix: [0.0204377  0.14902438]


In [ ]:
# this cell
import pandas as pd
import numpy as np
import plotly.graph_objects as go
from scipy.linalg import expm
from scipy.optimize import minimize

## Estimated Parameters

Buraschi Estimations

In [ ]:
# Monthly estimations
c=1
M_B= np.array([[-1.122,0.747], [0.884, -0.888]])
Q_B= np.array([[0.160, 0.083], [-0.021, 0.009]])
rho_bar_B= np.array([-0.279, -0.247])
lambda_vector_B= np.array([4.612, 2.891])

Q_B= c * Q_B
M_B= c**2 * M_B


rho_bar_B= np.matrix.transpose(np.asmatrix(rho_bar_B))
M_B= np.asmatrix(M_B)
Q_B= np.asmatrix(Q_B)
lambda_vector_B= np.asmatrix(lambda_vector_B).T

beta_B=10
def sig_inf_fun(sig_inf, M,Q,beta):
  sig_inf= np.array([[sig_inf[0], sig_inf[1]], [sig_inf[2], sig_inf[3]]])
  return np.linalg.norm(beta * Q.T @ Q + M @ sig_inf + sig_inf @ M.T)
sig_inf_0=[0,0,0,0]
result= minimize(sig_inf_fun,sig_inf_0, args=(M_B,Q_B,beta_B))
Sigma_inf_B= np.asmatrix(np.array([[0.3741894 , 0.38773162], [0.38773159, 0.42523051]]))
def sig_inf_matlab(M,Q,beta):
  Omega_aux=beta * Q.T @ Q
  #A_aux= np.array([[2*M[0,0], 2*M[0,1], 0], [M[1,0], M[0,0]+M[1,1], M[0,1]], [0, 2*M[0,1], 2*M[1,1]]]) # Wrong code
  A_aux= np.array([[2*M[0,0], 2*M[0,1], 0], [M[1,0], M[0,0]+M[1,1], M[0,1]], [0, 2*M[1,0], 2*M[1,1]]]) # Corrected code
  b_aux= np.array([Omega_aux[0,0], Omega_aux[0,1], Omega_aux[1,1]])
  sig_inf= np.linalg.inv(A_aux) @ (-b_aux)
  return sig_inf

# Computing M
def M_inf_fun(M, sig_inf,Q,beta):
  M= np.array([[M[0], M[1]], [M[2], M[3]]])
  return np.linalg.norm(beta * Q.T @ Q + M @ sig_inf + sig_inf @ M.T)
M_0=[0,0,0,0]
#result= minimize(M_inf_fun,M_0, args=(sig_inf_B,Q_B,beta_B))

def init_M_Q_fun(init_M_Q, Sigma_0):
  M= np.array([[-1, init_M_Q[0]], [init_M_Q[1], -1]])
  Q= np.array([[init_M_Q[2], 0], [0, init_M_Q[3]]])
  beta= 10
  return np.linalg.norm(beta * Q.T @ Q + M @ Sigma_0 + Sigma_0 @ M.T)
init_M_Q_0= np.array([0, 0, 0, 0])
#result= minimize(init_M_Q_fun,init_M_Q_0, args=(Sigma_0))

In [ ]:
M= np.array([[0.7011,-0.4322], [0.2169, -0.1925]])
Q= np.array([[0.2044, 0.0273], [-0.0308, 0.0125]])
rho_bar= np.array([-0.3787, -0.1828])
beta= 7.7194
sig_inf_matlab(M,Q,beta)
result= minimize(sig_inf_fun,sig_inf_0, args=(M,Q,beta))
result.x

array([-0.16089222,  0.12058226,  0.12058246,  0.15394248])

In [ ]:
sig_inf_matlab(M_B,Q_B,beta_B)


array([0.37418953, 0.3877318 , 0.42523076])

In [ ]:
sig_inf= np.array([0.3741894 , 0.38773162, 0.38773159, 0.42523051])
sig_inf_fun(sig_inf, M_B,Q_B,beta_B)

1.6776851039606286e-07

In [ ]:
Sigma_inf_B @ lambda_vector_B

matrix([[2.84669363],
        [3.0175595 ]])

In [ ]:
# Estimation for SP-AAPL with Final CF= 3.413256e-02
M_B= np.array([[-3.8372,0.6828], [0.2424, -1.9120]])
Q_B= np.array([[-0.0252, -0.1505], [0.1111, 0.0293]])
rho_bar_B= np.array([0.1385, -0.1982])
Sigma_inf_B= np.array([[0.0222, 0.0228], [0.0228, 0.0740]])
mu_B= np.array([0.0983, 0.3700])
r=0.02
lambda_vector_B= np.linalg.solve(Sigma_inf_B,mu_B-r)

rho_bar_B= np.matrix.transpose(np.asmatrix(rho_bar_B))
M_B= np.asmatrix(M_B)
Q_B= np.asmatrix(Q_B)
lambda_vector_B= np.asmatrix(lambda_vector_B).T

beta_B=10

# Computing Sigma_inf
def sig_inf_fun(sig_inf, M,Q,beta):
  sig_inf= np.array([[sig_inf[0], sig_inf[1]], [sig_inf[2], sig_inf[3]]])
  return np.linalg.norm(beta * Q.T @ Q + M @ sig_inf + sig_inf @ M.T)
sig_inf_0=[0,0,0,0]
result= minimize(sig_inf_fun,sig_inf_0, args=(M_B,Q_B,beta_B))

#Sigma_inf_B= np.asmatrix(np.array([[0.3741894 , 0.38773162], [0.38773159, 0.42523051]]))

In [ ]:
# Estimation for SP-AAPL with Final CF= 3.376417e-02
#M_B= np.array([[-3.8324,0.6836], [0.2519, -1.9129]]) # original values
M_B= np.array([[-1.122,0.747], [0.884, -0.888]]) # test M
#Q_B= np.array([[-0.0237, -0.1497], [0.1109, 0.0298]]) # original values
Q_B= np.array([[0.3, 0.5], [-0.4, 0.9]]) # test Q
rho_bar_B= np.array([0.1267, -0.2033])
Sigma_inf_B= np.array([[0.0221, 0.0226], [0.0226, 0.0738]])
mu_B= np.array([0.0983, 0.3700])
r=0.02
lambda_vector_B= np.linalg.solve(Sigma_inf_B,mu_B-r)

rho_bar_B= np.matrix.transpose(np.asmatrix(rho_bar_B))
M_B= np.asmatrix(M_B)
Q_B= np.asmatrix(Q_B)
lambda_vector_B= np.asmatrix(lambda_vector_B).T

beta_B=10
def sig_inf_fun(sig_inf, M,Q,beta):
  sig_inf= np.array([[sig_inf[0], sig_inf[1]], [sig_inf[2], sig_inf[3]]])
  return np.linalg.norm(beta * Q.T @ Q + M @ sig_inf + sig_inf @ M.T)
sig_inf_0=[0,0,0,0]
result= minimize(sig_inf_fun,sig_inf_0, args=(M_B,Q_B,beta_B))
#Sigma_inf_B= np.asmatrix(np.array([[0.3741894 , 0.38773162], [0.38773159, 0.42523051]]))

In [ ]:
# Estimation for SP-AAPL with Final CF= 2.345108e-02 starting from Buraschi estimates by aa=0.6
M_B= np.array([[-1.7935,0.2995], [0.8630, -0.3569]]) # original values
#M_B= np.array([[-1.122,0.747], [0.884, -0.888]]) # test M
Q_B= np.array([[0.1310, 0.0333], [-0.0329, 0.0141]]) # original values
#Q_B= np.array([[0.3, 0.5], [-0.4, 0.9]]) # test Q
rho_bar_B= np.array([-0.4388, -0.2252])
Sigma_inf_B= np.array([[0.0237, 0.0202], [0.0202, 0.0243]])
mu_B= np.array([0.0983, 0.3700])
r=0.04
lambda_vector_B= np.linalg.solve(Sigma_inf_B,mu_B-r)

rho_bar_B= np.matrix.transpose(np.asmatrix(rho_bar_B))
M_B= np.asmatrix(M_B)
Q_B= np.asmatrix(Q_B)
lambda_vector_B= np.asmatrix(lambda_vector_B).T

beta_B=4.0074
def sig_inf_fun(sig_inf, M,Q,beta):
  sig_inf= np.array([[sig_inf[0], sig_inf[1]], [sig_inf[2], sig_inf[3]]])
  return np.linalg.norm(beta * Q.T @ Q + M @ sig_inf + sig_inf @ M.T)
sig_inf_0=[0,0,0,0]
result= minimize(sig_inf_fun,sig_inf_0, args=(M_B,Q_B,beta_B))
#Sigma_inf_B= np.asmatrix(np.array([[0.3741894 , 0.38773162], [0.38773159, 0.42523051]]))

In [ ]:
result.x

array([0.0249938 , 0.02761959, 0.02761961, 0.07412708])

In [ ]:
sighat_B=0.2 * Sigma_inf_B

# parameters for Shifted W
M_BS, Q_BS, rho_bar_BS, lambda_vector_BS= M_B, Q_B, rho_bar_B, lambda_vector_B
sighat_BS= sighat_B
Sigma_inf_BS= Sigma_inf_B

# paramteres for MMSW
M_BM, Q_BM, rho_bar_BM, lambda_vector_BM= M_B, Q_B, rho_bar_B, lambda_vector_B
sighat_BM= 0.7 * sighat_B
Sigma_inf_BM= sighat_BM + Sigma_inf_B

Estimation on S&P vs Appl

In [ ]:
# Estimated Parameters for S&P and Appl on Normal Period

M_N= np.array([[-2.9755, 0.5001], [0.2662, -1.2210]])
#M= np.array([[0, 0], [0, 0]])
Q_N= np.array([[-0.0216, -0.1560], [0.1106, 0.0292]])
#Q= np.array([[0, 0], [0, 0]])
#beta= 6 #beta=5.6347
#y_0= np.array([6.9755,-1.1141])
#y_0_0= np.array([0,0])
#y_0_0= np.log(np.array([5, 4]))

rho_bar_N= np.array([0.1252, -0.2814])
#rho= np.array([0, 0])
Sigma_inf_normal_N= np.array([[0.0150, 0.0176], [0.0176, 0.0653]])
Sigma_inf_total_N= np.array([[0.0370, 0.0400], [0.0400, 0.1107]])
#Sigma_inf_crisis= np.array([[0.0305, 0.0372], [0.0372, 0.0594]])
Sig_hat_total_vs_normal_N= np.subtract(Sigma_inf_total_N, Sigma_inf_normal_N)
#Sig_hat_crisis_vs_normal= np.subtract(Sigma_inf_crisis, Sigma_inf_normal)
#Matrix sigma^hat
sighat_N= Sig_hat_total_vs_normal_N
#sighat= 1/2 * (Sig_hat_total_vs_normal + np.transpose(Sig_hat_total_vs_normal))
#sighat= 0.001 * Sig_hat_total_vs_normal
#Sigma_0= Sigma_inf_normal


#T= 1
#T= 1/12
#T=1/52
#t_= 0

#tau= T - t_
#T_t= T - t_


# Matrix sigma^wishart
#sigmaw= Sigma_inf_normal


# risk free rate
r=0.02

lambda_bar_0= 1.0657
#lambda_0= 0.0001
lambda_bar_1= 3.76497
#lambda_01 and lambda_10
#lambda_01= lambda_bar_0 / (lambda_bar_0 + lambda_bar_1)
#lambda_10= lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)
daily_transmat= np.array([[0.99581131, 0.00418869],
       [0.01479807, 0.98520193]])


mu_N= np.array([0.1135, 0.4572])
#mu= np.array([r, r])

p=lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)


#K= 0.5


# Monte Carlo Parameters
#N= 252
#N= 5 * 8
#N=21*8
#paths= 10
#M_paths= 100
#M_paths= paths
#T_vec, dt = np.linspace(t_, T, N, retstep=True)
#dt= (T - t_) / N
#np.random.seed(seed=42)

rho_bar_N= np.matrix.transpose(np.asmatrix(rho_bar_N))
M_N= np.asmatrix(M_N)
Q_N= np.asmatrix(Q_N)

In [ ]:
#Estimated Wishart for Total Data for SP500 vs APPLE

M_T= np.array([[-3.8393, 0.7024], [0.2519, -1.9265]]) #
#M= np.array([[0, 0], [0, 0]])
Q_T= np.array([[-0.0243, -0.1491], [0.1085, 0.0321]]) #
#Q= np.array([[0, 0], [0, 0]])
#beta_T= 11
beta_T=10.9151 #
#y_0= np.array([6.9755,-1.1141]) #
#y_0_0= np.array([0,0])
#y_0_0= np.log(np.array([5, 4]))

rho_bar_T= np.array([0.1453, -0.1968])  #
#rho= np.array([0, 0])
Sigma_inf_normal_T= np.array([[0.0219, 0.0235], [0.0235, 0.0745]]) #
Sigma_inf_total_T= np.array([[0.0370, 0.0400], [0.0400, 0.1107]])
#Sigma_inf_crisis= np.array([[0.0305, 0.0372], [0.0372, 0.0594]])
Sig_hat_total_vs_normal_T= np.subtract(Sigma_inf_total_T, Sigma_inf_normal_T)
#Sig_hat_crisis_vs_normal= np.subtract(Sigma_inf_crisis, Sigma_inf_normal)
#Matrix sigma^hat
sighat_T= Sig_hat_total_vs_normal_T
#sighat= 1/2 * (Sig_hat_total_vs_normal + np.transpose(Sig_hat_total_vs_normal))
#sighat= 0.001 * Sig_hat_total_vs_normal
#Sigma_0= Sigma_inf_normal


#T= 1
#T= 1/12
#T=1/52
#t_= 0

#tau= T - t_
#T_t= T - t_


# Matrix sigma^wishart
#sigmaw= Sigma_inf_normal


# risk free rate
r=0.02

lambda_bar_0= 1.0657
#lambda_0= 0.0001
lambda_bar_1= 3.76497
#lambda_01 and lambda_10
#lambda_01= lambda_bar_0 / (lambda_bar_0 + lambda_bar_1)
#lambda_10= lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)
daily_transmat= np.array([[0.99581131, 0.00418869],
       [0.01479807, 0.98520193]])


mu_T= np.array([0.1135, 0.4572])
#mu= np.array([r, r])

p=lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)

# Monte Carlo Parameters
#N= 252
#N= 5 * 8
#N=21*8
#paths= 10
#M_paths= 100
#M_paths= paths
#T_vec, dt = np.linspace(t_, T, N, retstep=True)
#dt= (T - t_) / N
#np.random.seed(seed=42)

rho_bar_T= np.matrix.transpose(np.asmatrix(rho_bar_T))
M_T= np.asmatrix(M_T)
Q_T= np.asmatrix(Q_T)

SP-AAPL New estimation starting from Buraschi estimates

In [ ]:
# Estimated Parameters for S&P and Appl on Normal Period

M_N= np.array([[-1.7935, 0.2992], [0.9248, -0.3561]])
#M= np.array([[0, 0], [0, 0]])
Q_N= np.array([[0.0977, 0.0333], [-0.0333, 0.0142]])
#Q= np.array([[0, 0], [0, 0]])
beta_N= 4.0117 #beta=5.6347
#y_0= np.array([6.9755,-1.1141])
#y_0_0= np.array([0,0])
#y_0_0= np.log(np.array([5, 4]))

rho_bar_N= np.array([-0.4335, -0.1053])
#rho= np.array([0, 0])
mu_N= np.array([0.1135, 0.4572])
#mu= np.array([r, r])

#result= minimize(sig_inf_fun,sig_inf_0, args=(M_N,Q_N,beta_N))
#Sigma_inf_normal_N= result.x.reshape(2,2)
#Sigma_inf_normal_N= np.array([[0.0142, 0.0140], [0.0140, 0.0191]]) # Wrong Estimation from Matlab Code
Sigma_inf_normal_N= np.array([[0.0152, 0.0200], [0.0200, 0.0593]]) # Correct Estimation from Matlab Code
#Sigma_inf_normal_N= np.array([[0.00142, 0], [0, 0.00191]]) # Some value just to test
Sigma_inf_total_N= np.array([[0.0370, 0.0400], [0.0400, 0.1107]])
#Sigma_inf_total_N=np.array([[0.00143, 0], [0, 0.00192]]) # Some value just to test
#Sigma_inf_crisis= np.array([[0.0305, 0.0372], [0.0372, 0.0594]])
Sig_hat_total_vs_normal_N= np.subtract(Sigma_inf_total_N, Sigma_inf_normal_N)
#Sig_hat_crisis_vs_normal= np.subtract(Sigma_inf_crisis, Sigma_inf_normal)
#Matrix sigma^hat
sighat_N= Sig_hat_total_vs_normal_N
#sighat= 1/2 * (Sig_hat_total_vs_normal + np.transpose(Sig_hat_total_vs_normal))
#sighat= 0.001 * Sig_hat_total_vs_normal
#Sigma_0= Sigma_inf_normal


#T= 1
#T= 1/12
#T=1/52
#t_= 0

#tau= T - t_
#T_t= T - t_


# Matrix sigma^wishart
#sigmaw= Sigma_inf_normal


# risk free rate
r=0.04

lambda_bar_0= 1.0657
#lambda_0= 0.0001
lambda_bar_1= 3.76497
#lambda_01 and lambda_10
#lambda_01= lambda_bar_0 / (lambda_bar_0 + lambda_bar_1)
#lambda_10= lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)
daily_transmat= np.array([[0.99581131, 0.00418869],
       [0.01479807, 0.98520193]])


p=lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)


#K= 0.5


# Monte Carlo Parameters
#N= 252
#N= 5 * 8
#N=21*8
#paths= 10
#M_paths= 100
#M_paths= paths
#T_vec, dt = np.linspace(t_, T, N, retstep=True)
#dt= (T - t_) / N
#np.random.seed(seed=42)

rho_bar_N= np.matrix.transpose(np.asmatrix(rho_bar_N))
M_N= np.asmatrix(M_N)
Q_N= np.asmatrix(Q_N)

In [ ]:
#Estimated Wishart for Total Data for SP500 vs APPLE

M_T= np.array([[-1.7935, 0.2995], [0.8630, -0.3569]]) #
#M= np.array([[0, 0], [0, 0]])
Q_T= np.array([[0.1310, 0.0333], [-0.0329, 0.0141]]) #
#Q= np.array([[0, 0], [0, 0]])
#beta_T= 11
beta_T=4.0074#
#y_0= np.array([6.9755,-1.1141]) #
#y_0_0= np.array([0,0])
#y_0_0= np.log(np.array([5, 4]))

rho_bar_T= np.array([-0.4388, -0.2252])  #
#rho= np.array([0, 0])

mu_T= np.array([0.0983, 0.3700])
#mu= np.array([r, r])
#result= minimize(sig_inf_fun,sig_inf_0, args=(M_T,Q_T,beta_T))
#Sigma_inf_normal_T= result.x.reshape(2,2)
#Sigma_inf_normal_T= np.array([0.0249938,0.02761959,0.02761961,0.07412708]) # sigma infty obtained by optimization
Sigma_inf_normal_T= np.array([[0.0250,0.0276],[0.0276,0.0741]]) #Estimation from the corrected Matlab code
#Sigma_inf_normal_T= np.array([[0.0142, 0.001], [0.002, 0.00191]]) # Some value just to test
Sigma_inf_total_T= np.array([[0.0370, 0.0400], [0.0400, 0.1107]])
#Sigma_inf_total_T= np.array([[0.00142, 0], [0, 0.00191]]) # Some value just to test
#Sigma_inf_crisis= np.array([[0.0305, 0.0372], [0.0372, 0.0594]])
Sig_hat_total_vs_normal_T= np.subtract(Sigma_inf_total_T, Sigma_inf_normal_T)
#Sig_hat_crisis_vs_normal= np.subtract(Sigma_inf_crisis, Sigma_inf_normal)
#Matrix sigma^hat
sighat_T= Sig_hat_total_vs_normal_T
#sighat= 1/2 * (Sig_hat_total_vs_normal + np.transpose(Sig_hat_total_vs_normal))
#sighat= 0.001 * Sig_hat_total_vs_normal
#Sigma_0= Sigma_inf_normal


#T= 1
#T= 1/12
#T=1/52
#t_= 0

#tau= T - t_
#T_t= T - t_


# Matrix sigma^wishart
#sigmaw= Sigma_inf_normal


# risk free rate
r=0.04

lambda_bar_0= 1.0657
#lambda_0= 0.0001
lambda_bar_1= 3.76497
#lambda_01 and lambda_10
#lambda_01= lambda_bar_0 / (lambda_bar_0 + lambda_bar_1)
#lambda_10= lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)
daily_transmat= np.array([[0.99581131, 0.00418869],
       [0.01479807, 0.98520193]])


p=lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)

# Monte Carlo Parameters
#N= 252
#N= 5 * 8
#N=21*8
#paths= 10
#M_paths= 100
#M_paths= paths
#T_vec, dt = np.linspace(t_, T, N, retstep=True)
#dt= (T - t_) / N
#np.random.seed(seed=42)

rho_bar_T= np.matrix.transpose(np.asmatrix(rho_bar_T))
M_T= np.asmatrix(M_T)
Q_T= np.asmatrix(Q_T)

Estimations on S&P-AAPL starting from the mean of Buraschi estimation and the last estimation on S&P-AAPL

In [ ]:
# Estimated Parameters for S&P and Appl on Normal Period

#M_N= np.array([[-2.2018, 0.2812], [0.9243, -0.2925]])
#M= np.array([[0, 0], [0, 0]])
#M_N= np.array([[-2.53433997,  0.23701974],  [0.52893394, -0.12867877]  ])
#M_N= np.array([[-1, 0.1565], [1.46006, -1]]) # Fb 13 initial point
M_N= np.array([[-1.0202, 0.1418], [1.3596, -0.8224]]) # estimation Fb 14

#Q_N= np.array([[0.1444, 0.0336], [-0.0180, 0.0131]])
#Q_N= np.array([[0.0588, 0], [0, 0.11677]]) # Fb 13 initial point
Q_N= np.array([[0.0689, 0], [0, 0.1173]]) # estimation Fb 14

#Q= np.array([[0, 0], [0, 0]])
#beta_N= 2.9116 #beta=5.6347
#beta_N= 10 # Fb 13 initial point
beta_N= 7.5504 # estimation Fb 14

#y_0= np.array([6.9755,-1.1141])
#y_0_0= np.array([0,0])
#y_0_0= np.log(np.array([5, 4]))

#rho_bar_N= np.array([-0.5499, -0.1930]) # Fb 13 initial point
rho_bar_N= np.array([-0.5499, -0.1930]) # estimation Fb 14
#rho= np.array([0, 0])
#mu_N= np.array([0.0983, 0.3700]) # Fb 13 initial point
mu_N= np.array([0.1031, 0.4065]) # estimation Fb 14
#mu= np.array([r, r])

#result= minimize(sig_inf_fun,sig_inf_0, args=(M_N,Q_N,beta_N))
#Sigma_inf_normal_N= result.x.reshape(2,2)
#Sigma_inf_normal_N= np.array([[0.0142, 0.0140], [0.0140, 0.0191]]) # Wrong Estimation from Matlab Code
#Sigma_inf_normal_N= np.array([[0.0164, 0.0189], [0.0189, 0.0664]]) # Correct Estimation from Matlab Code
#Sigma_inf_normal_N= np.array([[0.02096, 0.0233], [0.0233, 0.1022]]) # Fb 13 initial point
Sigma_inf_normal_N= np.array([[0.0208, 0.0232], [0.0232, 0.1014]]) # estimation Fb 14
#Sigma_inf_normal_N= np.array([[0.02, 0.02], [0.02, 0.10]])
#Sigma_0= np.array([[0.0208, 0.0232], [0.0232, 0.1014]])
#Sigma_inf_normal_N= np.array([[0.00142, 0], [0, 0.00191]]) # Some value just to test
Sigma_inf_total_N= np.array([[0.0370, 0.0400], [0.0400, 0.1107]])
Sigma_inf_hist_N= np.array([[0.02096, 0.0233], [0.0233, 0.1022]])
#Sigma_inf_total_N=np.array([[0.00143, 0], [0, 0.00192]]) # Some value just to test
#Sigma_inf_crisis= np.array([[0.0305, 0.0372], [0.0372, 0.0594]])
Sig_hat_total_vs_normal_N= np.subtract(Sigma_inf_total_N, Sigma_inf_normal_N)
#Sig_hat_crisis_vs_normal= np.subtract(Sigma_inf_crisis, Sigma_inf_normal)
#Matrix sigma^hat
sighat_N= Sig_hat_total_vs_normal_N
#sighat= 1/2 * (Sig_hat_total_vs_normal + np.transpose(Sig_hat_total_vs_normal))
#sighat= 0.001 * Sig_hat_total_vs_normal
#Sigma_0= Sigma_inf_normal


#T= 1
#T= 1/12
#T=1/52
#t_= 0

#tau= T - t_
#T_t= T - t_


# Matrix sigma^wishart
#sigmaw= Sigma_inf_normal


# risk free rate
r=0.04

lambda_bar_0= 1.0657
#lambda_0= 0.0001
lambda_bar_1= 3.76497
#lambda_01 and lambda_10
#lambda_01= lambda_bar_0 / (lambda_bar_0 + lambda_bar_1)
#lambda_10= lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)
daily_transmat= np.array([[0.99581131, 0.00418869],
       [0.01479807, 0.98520193]])


p=lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)


Lambda= np.array([[1, 0],
       [0, 1]])
M_tild_N= M_N - Q_N @ Lambda
#K= 0.5


# Monte Carlo Parameters
#N= 252
#N= 5 * 8
#N=21*8
#paths= 10
#M_paths= 100
#M_paths= paths
#T_vec, dt = np.linspace(t_, T, N, retstep=True)
#dt= (T - t_) / N
#np.random.seed(seed=42)

rho_bar_N= np.matrix.transpose(np.asmatrix(rho_bar_N))
M_N= np.asmatrix(M_N)
Q_N= np.asmatrix(Q_N)
M_tild_N= np.asmatrix(M_tild_N)

In [ ]:
result= minimize(init_M_Q_fun,init_M_Q_0, args=(Sigma_inf_hist_N))
result.x

array([0.15652877, 1.46006019, 0.05884364, 0.11677385])

In [ ]:
Sigma_inf= np.array([[0.02, 0.02],  [0.02, 0.10]  ])
result= minimize(M_inf_fun,M_0, args=(Sigma_inf,Q_T,beta_T))
result.x

array([-2.53433997,  0.23701974,  0.52893394, -0.12867877])

In [ ]:
M_inf= np.array([[-2.53433997,  0.23701974],  [0.52893394, -0.12867877]  ])

In [ ]:
#Estimated Wishart for Total Data for SP500 vs APPLE

#M_T= np.array([[-2.0634, 0.3184], [0.9283, -0.3390]]) #
#M= np.array([[0, 0], [0, 0]])
#M_T= np.array([[-1.7808301 ,  0.31543275],  [0.63834585, -0.2463134]  ])
#M_T= np.array([[-1.7986,  0.3122],  [0.6319, -0.2488]  ])
#M_T= np.array([[-1, 0.202667], [1.5558, -1]]) # Feb 7
#M_T= np.array([[-1.0095, 0.1851], [1.4499, -0.9076]]) # Feb 11
#M_T= np.array([[-1.0485, 0.1901], [1.6126, -0.9787]]) # Feb 12 coef= 1
M_T= np.array([[-1.0027, 0.2022], [1.5593, -0.9996]]) # Feb 12 coef= 10 &  Feb 13 coef= 10
#Q_T= np.array([[0.1797, 0.0388], [-0.0199, 0.0111]]) #
#Q_T= np.array([[0.1779, 0.0384], [-0.0197, 0.0110]])
#Q= np.array([[0, 0], [0, 0]])
#Q_T= np.array([[0.076017, 0], [0, 0.09845]]) #Feb 7
#Q_T= np.array([[0.0673, 0], [0, 0.0876]]) #Feb 11
#Q_T= np.array([[0.0611, 0], [0, 0.0792]]) ## Feb 12 coef= 1
Q_T= np.array([[0.0769, 0], [0, 0.0992]]) ## Feb 12 coef= 10 &  Feb 13 coef= 10
#beta_T= 11
#beta_T=2.8112#
#beta_T=10 # Feb 7
#beta_T=7.8956 # Feb 11
#beta_T=9.8447 # Feb 12 coef= 1
beta_T=9.7316 # Feb 12 coef= 10 &  Feb 13 coef= 10
#beta_T=2.7831
#y_0= np.array([6.9755,-1.1141]) #
#y_0_0= np.array([0,0])
#y_0_0= np.log(np.array([5, 4]))

#rho_bar_T= np.array([-0.5076, -0.2169])  #
#rho_bar_T= np.array([-0.5125, -0.2151])
#rho_bar_T= np.array([-0.5181, -0.2187]) #Feb 11
#rho_bar_T= np.array([-0.4973, -0.2163]) # Feb 12 coef= 1
rho_bar_T= np.array([-0.5055, -0.2170]) # Feb 12 coef= 10 &  Feb 13 coef= 10
#rho= np.array([0, 0])

#mu_T= np.array([0.0983, 0.3700])
#mu_T= np.array([0.0799, 0.3151])
mu_T= np.array([0.0799, 0.3151]) #Feb 11, Feb 12 coef=10 &  Feb 13 coef= 10
#mu= np.array([r, r])
#result= minimize(sig_inf_fun,sig_inf_0, args=(M_T,Q_T,beta_T))
#Sigma_inf_normal_T= result.x.reshape(2,2)
#Sigma_inf_normal_T= np.array([0.0249938,0.02761959,0.02761961,0.07412708]) # sigma infty obtained by optimization
#Sigma_inf_normal_T= np.array([[0.0269,0.0301],[0.0301,0.0892]]) #Estimation from the corrected Matlab code
#Sigma_inf_normal_T= np.array([[0.0304,0.0322],[0.0322,0.0908]])
#Sigma_inf_normal_T= np.array([[0.0142, 0.001], [0.002, 0.00191]]) # Some value just to test
#Sigma_inf_normal_T= np.array([[0.0370, 0.0400], [0.0400, 0.1107]]) # 7 Feb
#Sigma_inf_normal_T= np.array([[0.02202, 0.0235], [0.0235, 0.0709]]) # Feb 11
#Sigma_inf_normal_T= np.array([[0.02189, 0.024099], [0.024099, 0.07125]]) # Feb 12 coef= 1
Sigma_inf_normal_T= np.array([[0.03669, 0.03966], [0.03966, 0.10977]]) # Feb 12 coef= 10 &  Feb 13 coef= 10
Sigma_inf_total_T= np.array([[0.0370, 0.0400], [0.0400, 0.1107]])
#Sigma_inf_total_T= np.array([[0.00142, 0], [0, 0.00191]]) # Some value just to test
#Sigma_inf_crisis= np.array([[0.0305, 0.0372], [0.0372, 0.0594]])
Sig_hat_total_vs_normal_T= np.subtract(Sigma_inf_total_T, Sigma_inf_normal_T)
#Sig_hat_crisis_vs_normal= np.subtract(Sigma_inf_crisis, Sigma_inf_normal)
#Matrix sigma^hat
sighat_T= Sig_hat_total_vs_normal_T
#sighat= 1/2 * (Sig_hat_total_vs_normal + np.transpose(Sig_hat_total_vs_normal))
#sighat= 0.001 * Sig_hat_total_vs_normal
#Sigma_0= Sigma_inf_normal


#T= 1
#T= 1/12
#T=1/52
#t_= 0

#tau= T - t_
#T_t= T - t_


# Matrix sigma^wishart
#sigmaw= Sigma_inf_normal


# risk free rate
r=0.04

lambda_bar_0= 1.0657
#lambda_0= 0.0001
lambda_bar_1= 3.76497
#lambda_01 and lambda_10
#lambda_01= lambda_bar_0 / (lambda_bar_0 + lambda_bar_1)
#lambda_10= lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)
daily_transmat= np.array([[0.99581131, 0.00418869],
       [0.01479807, 0.98520193]])


p=lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)

M_tild_T= M_T - Q_T @ Lambda
# Monte Carlo Parameters
#N= 252
#N= 5 * 8
#N=21*8
#paths= 10
#M_paths= 100
#M_paths= paths
#T_vec, dt = np.linspace(t_, T, N, retstep=True)
#dt= (T - t_) / N
#np.random.seed(seed=42)

rho_bar_T= np.matrix.transpose(np.asmatrix(rho_bar_T))
M_T= np.asmatrix(M_T)
Q_T= np.asmatrix(Q_T)
M_tild_T= np.asmatrix(M_tild_T)

In [ ]:
result= minimize(init_M_Q_fun,init_M_Q_0, args=(Sigma_inf_total_T))
result.x
#result.x= array([0.20266709, 1.55580834, 0.07601728, 0.09845571])

array([0.20266709, 1.55580834, 0.07601728, 0.09845571])

In [ ]:
sig_inf_matlab(M_T,Q_T,beta_T)

array([0.03669489, 0.03966136, 0.10977045])

In [ ]:
Sigma_inf= np.array([[0.032, 0.035],  [0.035, 0.10]  ])
result= minimize(M_inf_fun,M_0, args=(Sigma_inf,Q_T,beta_T))
result.x


array([-1.7808301 ,  0.31543275,  0.63834585, -0.2463134 ])

In [ ]:
M_inf= np.array([[-1.7808301 ,  0.31543275],  [0.63834585, -0.2463134]  ])

Estimation on NVIDIA vs INTEL

In [ ]:
# Parameters for NVIDIA vs INTEL on normal period

M_N= np.array([[-1.4379, 1.2629], [0.1591, -3.7249]])
Q_N= np.array([[-0.0230, -0.1499], [0.1079, 0.0326]])
beta_N= 15.0991 #beta=15.0991
#y_0= np.array([0.4982,2.9835])
#y_0_0= np.log(np.array([5, 4]))
rho_bar_N= np.array([0.1573, -0.1010])
#rho= np.array([0, 0])
Sigma_inf_normal_N= np.array([[0.0949, 0.0353], [0.0353, 0.0492]])
# To estimate
Sigma_inf_total_N= np.array([[0.2499, 0.0842], [0.0842, 0.1031]])
#Sigma_inf_crisis= np.array([[0.0305, 0.0372], [0.0372, 0.0594]])
Sig_hat_total_vs_normal_N= np.subtract(Sigma_inf_total_N, Sigma_inf_normal_N)
#Sig_hat_crisis_vs_normal= np.subtract(Sigma_inf_crisis, Sigma_inf_normal)
#Matrix sigma^hat
sighat_N= Sig_hat_total_vs_normal_N
#sighat= np.array([[0.0001, 0.0001], [0.0001, 0.0001]])

#Sigma_0= Sigma_inf_normal

#T= 1
#t_= 0
#tau= T- t_
#T_t= T- t_

# Matrix sigma^wishart
#sigmaw= Sigma_inf_normal

# risk free rate
r=0.04

lambda_bar_0= 2.7019
lambda_bar_1= 5.0988

#lambda_01= lambda_0 / (lambda_0 + lambda_1)
#lambda_10= lambda_1 / (lambda_0 + lambda_1)

mu_N= np.array([0.5797, 0.1167])
#mu= np.array([r, r])

p=lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)

#N=252
#paths= 10
#paths= 2000
#M_paths= 30
#T_vec, dt = np.linspace(t_, T, N, retstep=True)

#np.random.seed(seed=42)

rho_bar_N= np.matrix.transpose(np.asmatrix(rho_bar_N))
M_N= np.asmatrix(M_N)
Q_N= np.asmatrix(Q_N)

In [ ]:
# Estimated Wishart on Total Data for NVIDIA vs INTEL

#M_T= np.array([[-1.4263, 1.7276], [0.1493, -3.3176]]) #
M_T= np.array([[-0.55655486,  0.5210562],  [0.16692936, -1.91016308]  ])
Q_T= np.array([[-0.0241, -0.1542], [0.1089, 0.0335]]) #
beta_T= 15.9860 #beta=15.9860 #
#y_0= np.array([0.4982,2.9835]) #
#y_0_0= np.log(np.array([5, 4]))
rho_bar_T= np.array([0.1687, -0.1242]) #
#rho= np.array([0, 0])
#Sigma_inf_normal_T= np.array([[0.1323, 0.0517], [0.0517, 0.0623]]) #
Sigma_inf_normal_T= np.array([[0.24, 0.08], [0.08, 0.10]])
# To estimate
Sigma_inf_total_T= np.array([[0.2499, 0.0842], [0.0842, 0.1031]])
#Sigma_inf_crisis= np.array([[0.0305, 0.0372], [0.0372, 0.0594]])
Sig_hat_total_vs_normal_T= np.subtract(Sigma_inf_total_T, Sigma_inf_normal_T)
#Sig_hat_crisis_vs_normal= np.subtract(Sigma_inf_crisis, Sigma_inf_normal)
#Matrix sigma^hat
sighat_T= Sig_hat_total_vs_normal_T
#sighat= np.array([[0.0001, 0.0001], [0.0001, 0.0001]])

#Sigma_0= Sigma_inf_normal

#T= 1
#t_= 0
#tau= T- t_
#T_t= T- t_

# Matrix sigma^wishart
#sigmaw= Sigma_inf_normal

# risk free rate
r=0.04

lambda_bar_0= 2.7019
lambda_bar_1= 5.0988

#lambda_01= lambda_0 / (lambda_0 + lambda_1)
#lambda_10= lambda_1 / (lambda_0 + lambda_1)

mu_T= np.array([0.5797, 0.1167])
#mu= np.array([r, r])

p=lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)


#N=252
#paths= 10
#paths= 2000
#M_paths= 30
#T_vec, dt = np.linspace(t_, T, N, retstep=True)

#np.random.seed(seed=42)

rho_bar_T= np.matrix.transpose(np.asmatrix(rho_bar_T))
M_T= np.asmatrix(M_T)
Q_T= np.asmatrix(Q_T)

In [ ]:
Sigma_inf_W= np.array([[0.24, 0.08], [0.08, 0.10]])
result= minimize(M_inf_fun,M_0, args=(Sigma_inf_W,Q_N,beta_N))
result.x


array([-0.55655486,  0.5210562 ,  0.16692936, -1.91016308])

In [ ]:
M_inf= np.array([[-0.55655486,  0.5210562],  [0.16692936, -1.91016308]  ])

Estimated Nvidia-Intel starting from the mean point of Buraschi and first SP-AAPL estimations

In [ ]:
# this cell Marcos
# Parameters for NVIDIA vs INTEL on normal period

#M_N= np.array([[-1.8246, 0.2301], [0.3607, -0.3294]]) # 9 Dec
#M_N= np.array([[-0.83012384, -0.12237494],  [0.16116928, -0.1384248]  ])
#M_N= np.array([[-0.8384, -0.1236],  [0.1596, -0.1398]  ])
#M_N= np.array([[-1, 2.3357], [-0.4064, -1]]) # Initial point Feb 14
M_N= np.array([[-0.9777, 2.0966], [-0.4279, -1.0156]]) # Estimation Feb 17
#Q_N= np.array([[0.2309, 0.0237], [-0.0392, 0.0156]]) # 9 Dec
#Q_N= np.array([[0.2286, 0.0235], [-0.0388, 0.0154]])
#Q_N= np.array([[0.13414, 0], [0, 0.15946]]) # Initial point Feb 14
Q_N= np.array([[0.1403, 0], [0, 0.1681]]) # Estimation Feb 17
#beta_N= 6.9267 #beta=15.0991 9 Dec
#beta_N= 10 # Initial point Feb 14
beta_N= 10.2032  # Estimation Feb 17
#beta_N= 6.8574
#y_0= np.array([0.4982,2.9835])
#y_0_0= np.log(np.array([5, 4]))
#rho_bar_N= np.array([-0.4089, -0.1274]) # 9 Dec, # Initial point Feb 14
rho_bar_N= np.array([-0.4087, -0.1274]) # Estimation Feb 17
#rho_bar_N= np.array([-0.4049, -0.1285])
#rho= np.array([0, 0])
#Sigma_inf_normal_N= np.array([[0.1091, 0.0394], [0.0394, 0.0516]]) # 9 Dec
#Sigma_inf_normal_N= np.array([[0.2460, 0.0668], [0.0668, 0.1000]]) # Initial point Feb 14
Sigma_inf_normal_N= np.array([[0.2460, 0.0668], [0.0668, 0.1138]]) # Estimation Feb 17
#Sigma_inf_normal_N= np.array([[0.2479, 0.0836], [0.0836, 0.1023]])
#Sigma_inf_normal_N= np.array([[0.2114, 0.0572],  [0.0572, 0.0846] ])
# To estimate
Sigma_inf_hist_N= np.array([[0.2460, 0.0668], [0.0668, 0.1000]])
Sigma_inf_total_N= np.array([[0.2479, 0.0836], [0.0836, 0.1023]])
#Sigma_inf_crisis= np.array([[0.0305, 0.0372], [0.0372, 0.0594]])
Sig_hat_total_vs_normal_N= np.subtract(Sigma_inf_total_N, Sigma_inf_normal_N)
#Sig_hat_crisis_vs_normal= np.subtract(Sigma_inf_crisis, Sigma_inf_normal)
#Matrix sigma^hat
sighat_N= Sig_hat_total_vs_normal_N
#sighat= np.array([[0.0001, 0.0001], [0.0001, 0.0001]])

#Sigma_0= Sigma_inf_normal

#T= 1
#t_= 0
#tau= T- t_
#T_t= T- t_

# Matrix sigma^wishart
#sigmaw= Sigma_inf_normal

# risk free rate
r=0.04
Lambda= np.array([[1, 0],
       [0, 1]])

lambda_bar_0= 2.7019
lambda_bar_1= 5.0988

#lambda_01= lambda_0 / (lambda_0 + lambda_1)
#lambda_10= lambda_1 / (lambda_0 + lambda_1)

#mu_N= np.array([0.5797, 0.1167])
mu_N= np.array([0.4567, 0.0599])
#mu_N= np.array([0.0983, 0.3700])
#mu= np.array([r, r])

p=lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)
M_tild_N= M_N - Q_N @ Lambda

#N=252
#paths= 10
#paths= 2000
#M_paths= 30
#T_vec, dt = np.linspace(t_, T, N, retstep=True)

#np.random.seed(seed=42)

rho_bar_N= np.matrix.transpose(np.asmatrix(rho_bar_N))
M_N= np.asmatrix(M_N)
Q_N= np.asmatrix(Q_N)
M_tild_N= np.asmatrix(M_tild_N)

In [ ]:
result= minimize(init_M_Q_fun,init_M_Q_0, args=(Sigma_inf_hist_N))
result.x

array([ 2.33574912, -0.40640199,  0.13414318,  0.15946639])

In [ ]:
#sighat_MW= (lambda_bar_1/(lambda_bar_0 + lambda_bar_1))* sighat_N
#sighat_MW= Sigma_inf_normal_T - Sigma_inf_normal_N
Sigma_inf_MW= np.array([[0.22, 0.06],  [0.06, 0.09]  ])

In [ ]:
result= minimize(M_inf_fun,M_0, args=(Sigma_inf_MW,Q_N,beta_N))
result.x

array([-0.83012384, -0.12237494,  0.16116928, -0.1384248 ])

In [ ]:
M_inf= np.array([[-0.83012384, -0.12237494],  [0.16116928, -0.1384248]  ])

In [ ]:
M_inf_fun(np.array([-0.83012384, -0.12237494,  0.16116928, -0.1384248 ]), Sigma_inf_MW,Q_N,beta_N)

2.0931939593215624e-07

In [ ]:
# This cell Marcos here we need to insert Wishart parameters
# Estimated Wishart on Total Data for NVIDIA vs INTEL

#M_T= np.array([[-1.6422, 0.2642], [0.3835, -0.3764]]) # 9 Dec
#M_T= np.array([[-0.90176674, -0.12201229],  [0.24368087, -0.22960478]  ])
#M_T= np.array([[-0.9108, -0.1232],  [0.2412, -0.2319]  ])
#M_T= np.array([[-0.9468, -0.1281],  [0.2315, -0.2411]  ]) # fixing Q, changing M
#M_T= np.array([[-1.0001, 0.1924], [0.5949, -1.0002]]) # 23 Jan
#M_T= np.array([[-1.1, 0.1732], [0.6284, -1.1]]) # 24 Jan
#M_T= np.array([[-0.9496, 0.2032], [0.6054, -1.0952]]) # 25 Jan
#M_T= np.array([[-0.4051, 0.2786], [0.3756, -1.0489]]) # 28 Jan
#M_T= np.array([[-1, 0.25375], [0.56974, -1]]) # 29 Jan Python code estimation
#M_T= np.array([[-3.7414, 2.4110], [ 0.7180, -4.2906]]) # 4 Feb by Prof. Escobar
#M_T= np.array([[-1, 0.19238], [0.59507, -1]]) # Initial point Feb 14
M_T= np.array([[-0.9998, 0.1923], [0.5944, -0.9991]]) # Estimation Feb 17
#Q_T= np.array([[0.2271, 0.0248], [-0.0342, 0.0139]]) # 9 Dec
#Q_T= np.array([[0.2248, 0.0246], [-0.0339, 0.0138]])
#Q_T= np.array([[0.2157, 0.0236], [-0.0325, 0.0132]]) # fixing M, changing Q
#Q_T= np.array([[0.2153, 0], [0, 0.1025]]) # 23 Jan
#Q_T= np.array([[0.2153, 0], [0, 0.10251]]) # Initial point Feb 14
Q_T= np.array([[0.2159, 0], [0, 0.1028]]) # Estimation Feb 17
#Q_T= np.array([[0.1938, 0], [0, 0.0923]]) # 24 Jan
#Q_T= np.array([[0.1942, 0], [0, 0.0925]]) # 25 Jan
#Q_T= np.array([[0.1613, 0], [0, 0.1594]]) # 28 Jan
#Q_T= np.array([[0.2129, 0], [0, 0.10457]]) ## 29 Jan Python code estimation
#Q_T= np.array([[0.4659, 0], [0, 0.2381]]) # 4 Feb by Prof. Escobar
#beta_T= 8.5767 #beta=15.9860 # 9 Dec
#beta_T= 10 #beta=15.9860 # 23 Jan, Initila point Feb 14
beta_T= 9.9408 # Estimation Feb 17
#beta_T= 9 #24 Jan
#beta_T= 9.0164 #24 Jan
#beta_T= 4.0316 #28 Jan
#beta_T=9.99786  # 29 Jan Python code estimation
#beta_T= 7.8563  # 4 Feb by Prof. Escobar
#beta_T=8.4909
#y_0= np.array([0.4982,2.9835]) #
#y_0_0= np.log(np.array([5, 4]))
#rho_bar_T= np.array([-0.3869, -0.1874]) # 9 Dec, 23 Jan
#rho_bar_T= np.array([-0.3482, -0.2061]) # 24 Jan
#rho_bar_T= np.array([-0.3594, -0.1931]) # 25 Jan
#rho_bar_T= np.array([-0.2848, -0.2831]) # 28 Jan
#rho_bar_T= np.array([-0.3869, -0.1874]) # 29 Jan Python code estimation
#rho_bar_T= np.array([-0.3354, -0.1962]) # 4 Feb by Prof. Escobar
#rho_bar_T= np.array([-0.3831, -0.1891]) #
rho_bar_T= np.array([-0.3865, -0.1874]) # Estimation Feb 17
#rho= np.array([0, 0])
#Sigma_inf_normal_T= np.array([[0.1472, 0.0589], [0.0589, 0.0692]]) # 9 Dec
#Sigma_inf_normal_T= np.array([[0.2479, 0.0836], [0.0836, 0.1023]])
#Sigma_inf_normal_T= np.array([[0.2307, 0.0761],  [0.0761, 0.0937]  ])
#Sigma_inf_normal_T= np.array([[0.2166, 0.0722], [0.0722, 0.0903]])
#Sigma_inf_normal_T= np.array([[0.2290, 0.0728], [0.0728, 0.0843]])
#Sigma_inf_normal_T= np.array([[0.2478, 0.0835], [0.0835, 0.1023]]) # 23 Jan
#Sigma_inf_normal_T= np.array([[0.1617, 0.0512], [0.0512, 0.0641]]) # 24 Jan
#Sigma_inf_normal_T= np.array([[0.1928, 0.0641], [0.0641, 0.0707]]) # 25 Jan
#Sigma_inf_normal_T= np.array([[0.1686, 0.0568], [0.0568, 0.0692]]) # 28 Jan
#Sigma_inf_normal_T= np.array([[0.25386, 0.04029], [0.04029, 0.058646]]) # 4 Feb by Prof. Escobar Long term covariance
#Sigma_inf_normal_T= np.array([[0.2479, 0.0836], [0.0836, 0.1023]]) # Initila point Feb 14
Sigma_inf_normal_T= np.array([[0.2479, 0.0836], [0.0836, 0.1023]]) # Estimation Feb 17
# To estimate
Sigma_inf_total_T= np.array([[0.2479, 0.0836], [0.0836, 0.1023]])
#Sigma_inf_crisis= np.array([[0.0305, 0.0372], [0.0372, 0.0594]])
Sig_hat_total_vs_normal_T= np.subtract(Sigma_inf_total_T, Sigma_inf_normal_T)
#Sig_hat_crisis_vs_normal= np.subtract(Sigma_inf_crisis, Sigma_inf_normal)
#Matrix sigma^hat
sighat_T= Sig_hat_total_vs_normal_T
#sighat= np.array([[0.0001, 0.0001], [0.0001, 0.0001]])

#Sigma_0= Sigma_inf_normal

#T= 1
#t_= 0
#tau= T- t_
#T_t= T- t_

# Matrix sigma^wishart
#sigmaw= Sigma_inf_normal

# risk free rate
r=0.04

lambda_bar_0= 2.7019
lambda_bar_1= 5.0988

#lambda_01= lambda_0 / (lambda_0 + lambda_1)
#lambda_10= lambda_1 / (lambda_0 + lambda_1)

mu_T= np.array([0.3170, 0.0400])
#mu= np.array([r, r])

p=lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)
M_tild_T= M_T - Q_T @ Lambda

#N=252
#paths= 10
#paths= 2000
#M_paths= 30
#T_vec, dt = np.linspace(t_, T, N, retstep=True)

#np.random.seed(seed=42)

rho_bar_T= np.matrix.transpose(np.asmatrix(rho_bar_T))
M_T= np.asmatrix(M_T)
Q_T= np.asmatrix(Q_T)
M_tild_T= np.asmatrix(M_tild_T)

In [ ]:
sig_inf_matlab(M_T,Q_T,beta_T)

array([0.25386569, 0.04029772, 0.05864609])

In [ ]:
Sigma_inf= np.array([[0.24, 0.08],  [0.08, 0.10]  ])

In [ ]:
result= minimize(M_inf_fun,M_0, args=(Sigma_inf_total_T,Q_T,beta_T))

In [ ]:
result.x

array([-1.01278245,  0.23083955,  0.57902556, -0.98668381])

In [ ]:
M_inf_fun(np.array([-0.90176674, -0.12201229,  0.24368087, -0.22960478]), Sigma_inf,Q_T,beta_T)

1.802335374883666e-07

In [ ]:
M_inf= np.array([[-0.90176674, -0.12201229],  [0.24368087, -0.22960478]  ])

In [ ]:
Sigma_inf_total_T

array([[0.2479, 0.0836],
       [0.0836, 0.1023]])

In [ ]:
result= minimize(init_M_Q_fun,init_M_Q_0, args=(Sigma_inf_total_T))
result.x

array([0.19238994, 0.59507708, 0.2153214 , 0.10251937])

# GOOG-INTC estimation

In [ ]:
# Parameters for GOOG vs INTEL on normal period

#M_N= np.array([[-1, -0.8392], [1.7225, -1]]) # Initial point Feb 20
M_N= np.array([[-1.0007, -0.8390], [1.7223, -1.0006]]) # Est Feb 22

#Q_N= np.array([[0.15217, 0], [0, 0.0923]]) # Initial point Feb 22
Q_N= np.array([[0.1521, 0], [0, 0.0923]]) # Est Feb 22

#beta_N= 10  # Initial Point Feb 20
beta_N= 9.9356  # Initial Point Feb 22
#y_0= np.array([0.9124,2.5416]) # Est Feb 22

#rho_bar_N= np.array([-0.4087, -0.1274]) # Initial point Feb 20
rho_bar_N= np.array([-0.4081, -0.1274]) # Est Feb 22

#Sigma_inf_normal_N= np.array([[0.0877, 0.03346], [0.03346, 0.10026]]) # Initial point Feb 20
Sigma_inf_normal_N= np.array([[0.0870, 0.0332], [0.0332, 0.0995]]) # Est Feb 22

Sigma_inf_hist_N= np.array([[0.0877, 0.03346], [0.03346, 0.10026]])
Sigma_inf_total_N= np.array([[0.0927, 0.0424], [0.0424, 0.1107]])

Sig_hat_total_vs_normal_N= np.subtract(Sigma_inf_total_N, Sigma_inf_normal_N)

sighat_N= Sig_hat_total_vs_normal_N

r=0.04
Lambda= np.array([[1, 0],
       [0, 1]])

lambda_bar_0= 44.003
lambda_bar_1= 128.685

mu_N= np.array([0.2207, 0.0349])


p=lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)
M_tild_N= M_N - Q_N @ Lambda


rho_bar_N= np.matrix.transpose(np.asmatrix(rho_bar_N))
M_N= np.asmatrix(M_N)
Q_N= np.asmatrix(Q_N)
M_tild_N= np.asmatrix(M_tild_N)

In [ ]:

# Estimated Wishart on Total Data for GOOG vs INTEL

#M_T= np.array([[-1, -0.44377], [1.4447, -1]]) # Initial point Feb 20
M_T= np.array([[-0.9990, -0.4437], [1.4443, -1]]) # Estimation Feb 22

#Q_T= np.array([[0.1493, 0], [0, 0.09944]]) # Initial point Feb 20
Q_T= np.array([[0.1494, 0], [0, 0.0995]]) # Estimation Feb 22

#beta_T= 10 # Initial point Feb 20
beta_T= 9.9110 # Initial point Feb 22
#y_0= np.array([0.9124,2.5416]) # Est Feb 22

#rho_bar_T= np.array([-0.3865, -0.1874]) # Estimation Feb 17
rho_bar_T= np.array([-0.3860, -0.1874]) # Estimation Feb 22

#Sigma_inf_normal_T= np.array([[0.0927, 0.0424], [0.0424, 0.1107]])  # Initial point Feb 20
Sigma_inf_normal_T= np.array([[0.0920, 0.0421], [0.0421, 0.1099]])  # Estimation Feb 22

Sigma_inf_total_T= np.array([[0.0927, 0.0424], [0.0424, 0.1107]])
#Sigma_inf_crisis= np.array([[0.0305, 0.0372], [0.0372, 0.0594]])
Sig_hat_total_vs_normal_T= np.subtract(Sigma_inf_total_T, Sigma_inf_normal_T)

sighat_T= Sig_hat_total_vs_normal_T

r=0.04

lambda_bar_0= 44.003
lambda_bar_1= 128.685


mu_T= np.array([0.2090, 0.0330])

p=lambda_bar_1 / (lambda_bar_0 + lambda_bar_1)
M_tild_T= M_T - Q_T @ Lambda


rho_bar_T= np.matrix.transpose(np.asmatrix(rho_bar_T))
M_T= np.asmatrix(M_T)
Q_T= np.asmatrix(Q_T)
M_tild_T= np.asmatrix(M_tild_T)

In [ ]:
result= minimize(init_M_Q_fun,init_M_Q_0, args=(Sigma_inf_hist_N))
result.x

array([-0.8392663 ,  1.7225184 ,  0.15217216,  0.09233045])

## Data Management

In [ ]:
# This cell Marcos
c=1
Q_N, M_N= c* Q_N, c**2 * M_N
Q_T, M_T= c* Q_T, c**2 * M_T

# parameters for Merton process
Sigma_inf_M= Sigma_inf_total_T
lambda_vector_M = np.asmatrix(np.linalg.solve(Sigma_inf_M,mu_T-r)).T
#lambda_vector_M = lambda_vector_B

# parameters for Wishart process
M_W, Q_W, rho_bar_W= M_T, Q_T, rho_bar_T
Sigma_inf_W= Sigma_inf_normal_T
lambda_vector_W= np.asmatrix(np.linalg.solve(Sigma_inf_W,mu_T-r)).T
#M_W, Q_W, rho_bar_W= M_B, Q_B, rho_bar_B
#Sigma_inf_W= Sigma_inf_B
#lambda_vector_W= lambda_vector_B


# parameters for Shifted Wishart process
M_SW, Q_SW, rho_bar_SW= M_T, Q_T, rho_bar_T
sighat_SW= sighat_T
#sighat_SW= 0.5 * Sigma_inf_normal_T
Sigma_inf_SW= sighat_SW + Sigma_inf_normal_T  #Sigma_inf_SW= Sigma_inf_total_T
#Sigma_inf_SW= Sigma_inf_normal_T
lambda_vector_SW= np.asmatrix(np.linalg.solve(Sigma_inf_SW,mu_T-r)).T

# parameters for Markov Shifted Process
M_MW, Q_MW, rho_bar_MW= M_N, Q_N, rho_bar_N
#sighat_MW= (lambda_bar_1/(lambda_bar_0 + lambda_bar_1))* sighat_N
sighat_MW= ((lambda_bar_0 + lambda_bar_1)/lambda_bar_0)* sighat_N
#sighat_MW= Sigma_inf_normal_T - Sigma_inf_normal_N
Sigma_inf_MW= sighat_N + Sigma_inf_normal_N
#Sigma_inf_MW= Sigma_inf_normal_N
#lambda_vector_MW= np.asmatrix(np.linalg.solve(Sigma_inf_MW,mu_N-r)).T
lambda_vector_MW= np.asmatrix(np.linalg.solve(Sigma_inf_MW,mu_T-r)).T

In [ ]:
lambda_vector_SW

array([[0.0007, 0.0003],
       [0.0003, 0.0008]])

In [ ]:
Sigma_inf_MW

array([[0.0927, 0.0424],
       [0.0424, 0.1107]])

## Functions

In [ ]:
# This cell Marcos
# Constants and input matrices
delta = -5  # Risk aversion coefficient
#lambda_vector = np.asmatrix(np.array([0.1, 0.2])).T  # Example values for lambda
#lambda_vector = np.asmatrix(np.array([4.612, 2.891])).T # Estimated values from Buraschi
#lambda_vector = lambda_vector_W

# Sigma and widehat_sigma matrices for SW and MMSW processes
#Sigma_SW = np.array([[0.2, 0.1], [0.1, 0.3]])  # For Shifted Wishart
#Sigma_MW = np.array([[0.2, 0.1], [0.1, 0.3]])  # For MMSW
#widehat_Sigma = np.array([[0.05, 0.02], [0.02, 0.07]])  # Constant shift matrix

# Define time horizon t
#t = 1.0  # Example time

# tau= T-t
#tau = 0.9

def B_t (lambda_vector,M,Q,rho_bar,tau, delta):
  #Q = np.array([[0.1, 0.2], [0.3, 0.4]])  # Example Q matrix
  #rho_bar = np.array([0.05, 0.1])  # Example correlation vector

  # Define matrices for B(t) computation
  #Gamma = np.array([[0.5, 0.2], [0.3, 0.4]])  # Example Gamma matrix
  #Lambda = np.array([[0.2, 0.1], [0.1, 0.2]])  # Example Lambda matrix
  #C = np.array([[0.3, 0.4], [0.1, 0.2]])  # Example C matrix
  Gamma= M + delta /(1-delta) * np.matrix.transpose(Q) @ rho_bar @ np.matrix.transpose(lambda_vector)
  C= 0.5 * delta / (1-delta)**2 * lambda_vector @ lambda_vector.T
  Lambda= np.matrix.transpose(Q) @ ((1-delta)* np.eye(2) + delta * rho_bar @ np.matrix.transpose(rho_bar)) @ Q

  # Compute the block matrix F_ij(t) = exp( t * A ), where A is a 4x4 matrix composed of Gamma, Lambda, and C
  A = np.block([[Gamma, -2 * Lambda], [C, -Gamma.T]])

  F = expm(tau * A)

  # Extract F_11, F_12, F_21, F_22 from F
  F_11 = F[:2, :2]
  F_12 = F[:2, 2:]
  F_21 = F[2:, :2]
  F_22 = F[2:, 2:]

  # Compute B(t)
  return np.linalg.inv(F_22) @ F_21

# Myopic Portfolio
def portfolio_allocation_M(lambda_vector_M, delta):
  return (1 / (1 - delta)) * lambda_vector_M

# Wishart Portfolio after Buraschi
def portfolio_allocation_W(lambda_vector_W, M_W,Q_W,rho_bar_W,tau, delta):
  return (1 / (1 - delta)) * lambda_vector_W + 2 * B_t (lambda_vector_W, M_W,Q_W,rho_bar_W,tau, delta) @ Q_W.T @ rho_bar_W

# Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
def portfolio_allocation_SW(lambda_vector_SW, M_SW,Q_SW,rho_bar_SW,Sigma_inf_SW,sighat_SW,tau, delta):
  inv_Sigma_SW = np.linalg.inv(Sigma_inf_SW)
  output= (1 / (1 - delta)) * lambda_vector_SW + 2 * B_t (lambda_vector_SW, M_SW,Q_SW,rho_bar_SW,tau, delta) @ Q_SW.T @ rho_bar_SW - 2 * inv_Sigma_SW @ sighat_SW @ B_t (lambda_vector_SW, M_SW,Q_SW,rho_bar_SW,tau, delta) @ Q_SW.T @ rho_bar_SW
  return output

# Theorem 2: Optimal Portfolio Strategy under MMSW Process
# Simulating a Markov chain that switches between states 0 and 1
# If state is 0: widehat_Sigma is zero, if state is 1: widehat_Sigma is the matrix above
#p=0.7
# Example: Let's simulate the Markov chain switching between two states
#state = np.random.choice([0, 1])  # Simulating the current state of the Markov chain
#p=0.7
#transition_matrix = np.array([[p, 1-p], [1-p, p]])
#current_state= 0
#state = np.random.choice(2, p=transition_matrix[current_state])

def portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, state, tau, delta):
# Use the state-dependent widehat_Sigma_MC in the MMSW process
  widehat_Sigma_MC = sighat_MW if state == 1 else np.zeros_like(sighat_MW)
  inv_Sigma_MW = np.linalg.inv(Sigma_inf_MW)
  output=  (1 / (1 - delta)) * lambda_vector_MW + 2 * B_t(lambda_vector_MW, M_MW,Q_MW,rho_bar_MW,tau, delta) @ Q_MW.T @ rho_bar_MW - 2 * inv_Sigma_MW @ widehat_Sigma_MC @ B_t(lambda_vector_MW,M_MW,Q_MW,rho_bar_MW,tau, delta) @ Q_MW.T @ rho_bar_MW
  return output

# Numerical Results

## SW and MMSW Comparison

In [ ]:
tau=1
state= 0
print("Merton: Optimal Portfolio Strategy under BS Process")
print(portfolio_allocation_M(lambda_vector_M))

print("Buraschi: Optimal Portfolio Strategy under Wishart Process")
print(portfolio_allocation_W(lambda_vector_W, M_W,Q_W,rho_bar_W,tau))

print("Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process")
print(portfolio_allocation_SW(lambda_vector_SW, M_SW,Q_SW,rho_bar_SW,Sigma_inf_SW,sighat_SW,tau))

print("\nTheorem 2: Optimal Portfolio Strategy under MMSW Process")
print(portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, state, tau))


Merton: Optimal Portfolio Strategy under BS Process
[[-0.43985937]
 [ 0.57311992]]
Buraschi: Optimal Portfolio Strategy under Wishart Process
[[-0.44118458]
 [ 0.58597053]]
Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
[[-0.43766162]
 [ 0.58098123]]

Theorem 2: Optimal Portfolio Strategy under MMSW Process
[[-0.43845766]
 [ 0.58308735]]


## Portfolio Allocation Over Time

In [ ]:
# This cell Marcos
# Define time horizon range t for tests
time_horizon = np.linspace(0, 5, 100)

# Compute portfolio allocations for both Theorems across the time horizon
portfolio_allocations_M = []
portfolio_allocations_W = []
portfolio_allocations_SW = []
portfolio_allocations_MMSW_S0 = []
portfolio_allocations_MMSW_S1 = []

for tau in time_horizon:
    # Compute the block matrix F_ij(t) = exp( t * A )
    #A = np.block([[Gamma, -2 * Lambda], [C, -Gamma.T]])
    #F = expm(t * A)

    # Extract F_11, F_12, F_21, F_22 from F
    #F_21 = F[2:, :2]
    #F_22 = F[2:, 2:]

    # Compute B(t)
    #B_t = np.linalg.inv(F_22) @ F_21
    # Merton Portfolio Strategy under Black Scholes Process
    portfolio_allocation_M_V= portfolio_allocation_M(lambda_vector_M, delta)
    #portfolio_allocation_M_V= portfolio_allocation_M(lambda_vector_B)

    # Buratchi Portfolio Strategy under Wishart Process
    portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_W, M_W,Q_W,rho_bar_W,tau, delta)
    #portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_B, M_B,Q_B,rho_bar_B,tau)

    # Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
    #inv_Sigma_SW = np.linalg.inv(Sigma_SW)
    portfolio_allocation_SW_V = portfolio_allocation_SW(lambda_vector_SW, M_SW,Q_SW,rho_bar_SW,Sigma_inf_SW,sighat_SW,tau, delta)
    #portfolio_allocation_SW_V = portfolio_allocation_SW(lambda_vector_B, M_BS,Q_BS,rho_bar_BS,Sigma_inf_BS,sighat_BS, tau)

    # Theorem 2: Optimal Portfolio Strategy under MMSW Process
    # Simulating a Markov chain that switches between states 0 and 1
    #state = np.random.choice([0, 1])  # Simulating the current state of the Markov chain
    #state = np.random.choice(2, p=transition_matrix[current_state])
    # The state-dependent widehat_Sigma
    #widehat_Sigma_MC = widehat_Sigma if state == 1 else np.zeros_like(widehat_Sigma)

    # Use the state-dependent widehat_Sigma_MC in the MMSW process
    #inv_Sigma_MW = np.linalg.inv(Sigma_MW)
    # MW portfolio for normal state
    portfolio_allocation_MMSW_V_S0 = portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, 0, tau, delta)
    #portfolio_allocation_MMSW_V_S0 = portfolio_allocation_MMSW(lambda_vector_B, M_BM, Q_BM, rho_bar_BM, Sigma_inf_BM,sighat_BM, 0, tau)
    # MW portfolio for Shifted State
    portfolio_allocation_MMSW_V_S1 = portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, 1, tau, delta)
    #portfolio_allocation_MMSW_V_S1 = portfolio_allocation_MMSW(lambda_vector_B, M_BM, Q_BM, rho_bar_BM, Sigma_inf_BM,sighat_BM, 1, tau)

    # Store the results
    portfolio_allocations_M.append(np.array(portfolio_allocation_M_V))
    portfolio_allocations_W.append(np.array(portfolio_allocation_W_V))
    portfolio_allocations_SW.append(np.array(portfolio_allocation_SW_V))
    portfolio_allocations_MMSW_S0.append(np.array(portfolio_allocation_MMSW_V_S0))
    portfolio_allocations_MMSW_S1.append(np.array(portfolio_allocation_MMSW_V_S1))

# Convert lists to numpy arrays for easier manipulation
#portfolio_allocations_SW = np.array(portfolio_allocations_SW)
#portfolio_allocations_MMSW_S0 = np.array(portfolio_allocations_MMSW_S0)
#portfolio_allocations_MMSW_S1 = np.array(portfolio_allocations_MMSW_S1)

# Create Plotly graphs to visualize the results
fig = go.Figure()

# Add traces for both components of the portfolio allocation under Merton
#fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='Merton'))
fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='Merton'))

# Add traces for both components of the portfolio allocation under Wishart
#fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_W)[:, 0], mode='lines', name='Wishart'))
fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_W)[:, 1], mode='lines', name='Wishart'))

# Add traces for both components of the portfolio allocation under Shifted Wishart
#fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_SW)[:, 0], mode='lines', name='Shifted W'))
fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_SW)[:, 1], mode='lines', name='Shifted W'))

# Add traces for both components of the portfolio allocation under MMSW State 0
#fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 0], mode='lines', name='MMSW Normal', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 1], mode='lines', name='MMSW Normal', line=dict(dash='dash')))

# Add traces for both components of the portfolio allocation under MMSW State 1
#fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 0], mode='lines', name='MMSW Crisis', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 1], mode='lines', name='MMSW Crisis', line=dict(dash='dash')))

# Customize the layout
#fig.update_layout(
#    #title="Portfolio Allocations under Merton, Wishart, Shifted Wishart and MMSW Processes",
#    title="Portfolio Allocations under Wishart Process",
#    xaxis_title="Time Horizon (T - t)",
#    yaxis_title="Portfolio Allocation",
#    legend_title="Portfolio Components",
#)

fig.write_html('portfolio_allocations_shifted_wishart_mmsw.html')

fig.update_layout(
#    title="Portfolio Allocations under Wishart Process",
    xaxis=dict(
        title="Time Horizon (T - t)",
        titlefont=dict(size=30),    # Font size of x-axis label
        tickfont=dict(size=30)      # Font size of x-axis tick numbers
    ),
    yaxis=dict(
        title="Portfolio Allocation",
        titlefont=dict(size=30),    # Font size of y-axis label
        tickfont=dict(size=30)      # Font size of y-axis tick numbers
    ),
    legend_title="Components",
    font=dict(size=30)  # General font (e.g., legend)
)

fig.write_html('portfolio_allocations_shifted_wishart_mmsw.html')

# Display the figure
fig.show()


# Sensitivity Analysis for $\lambda_0, \lambda_1$

In [ ]:
# Sensitivity wrt
# Define time horizon range t for tests
#time_horizon = np.linspace(0, 5, 100)
#lambda_0_space = np.linspace(0.2* lambda_bar_0, 5* lambda_bar_0, 100) # Sensitivity wrt lambda_0
lambda_1_space = np.linspace(0.2* lambda_bar_1, 5* lambda_bar_1, 100) # Sensitivity wrt lambda_1

tau= 5
# Compute portfolio allocations for both Theorems across the time horizon
portfolio_allocations_M = []
portfolio_allocations_W = []
portfolio_allocations_SW = []
portfolio_allocations_MMSW_S0 = []
portfolio_allocations_MMSW_S1 = []

#for lambda_0 in lambda_0_space: # Sensitivity wrt lambda_0
for lambda_1 in lambda_1_space: # Sensitivity wrt lambda_1
    # Compute the block matrix F_ij(t) = exp( t * A )
    #A = np.block([[Gamma, -2 * Lambda], [C, -Gamma.T]])
    #F = expm(t * A)

    # Extract F_11, F_12, F_21, F_22 from F
    #F_21 = F[2:, :2]
    #F_22 = F[2:, 2:]

    # Compute B(t)
    #B_t = np.linalg.inv(F_22) @ F_21
    # Merton Portfolio Strategy under Black Scholes Process
    portfolio_allocation_M_V= portfolio_allocation_M(lambda_vector_M, delta)
    #portfolio_allocation_M_V= portfolio_allocation_M(lambda_vector_B, delta)

    # Buratchi Portfolio Strategy under Wishart Process
    portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_W, M_W,Q_W,rho_bar_W,tau, delta)
    #portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_B, M_B,Q_B,rho_bar_B,tau, delta)

    # Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
    #inv_Sigma_SW = np.linalg.inv(Sigma_SW)
    portfolio_allocation_SW_V = portfolio_allocation_SW(lambda_vector_SW, M_SW,Q_SW,rho_bar_SW,Sigma_inf_SW,sighat_SW,tau, delta)
    #portfolio_allocation_SW_V = portfolio_allocation_SW(lambda_vector_B, M_BS,Q_BS,rho_bar_BS,Sigma_inf_BS,sighat_BS, tau, delta)

    # Theorem 2: Optimal Portfolio Strategy under MMSW Process
    # Simulating a Markov chain that switches between states 0 and 1
    #state = np.random.choice([0, 1])  # Simulating the current state of the Markov chain
    #state = np.random.choice(2, p=transition_matrix[current_state])
    # The state-dependent widehat_Sigma
    #widehat_Sigma_MC = widehat_Sigma if state == 1 else np.zeros_like(widehat_Sigma)

    #sighat_MW= ((lambda_0 + lambda_bar_1)/lambda_bar_1)* sighat_N # Sensitivity wrt lambda_0
    sighat_MW= ((lambda_bar_0 + lambda_1)/lambda_1)* sighat_N # Sensitivity wrt lambda_1
    # Use the state-dependent widehat_Sigma_MC in the MMSW process
    #inv_Sigma_MW = np.linalg.inv(Sigma_MW)
    # MW portfolio for normal state
    portfolio_allocation_MMSW_V_S0 = portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, 0, tau, delta)
    #portfolio_allocation_MMSW_V_S0 = portfolio_allocation_MMSW(lambda_vector_B, M_BM, Q_BM, rho_bar_BM, Sigma_inf_BM,sighat_BM, 0, tau, delta)
    # MW portfolio for Shifted State
    portfolio_allocation_MMSW_V_S1 = portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, 1, tau, delta)
    #portfolio_allocation_MMSW_V_S1 = portfolio_allocation_MMSW(lambda_vector_B, M_BM, Q_BM, rho_bar_BM, Sigma_inf_BM,sighat_BM, 1, tau, delta)

    # Store the results
    portfolio_allocations_M.append(np.array(portfolio_allocation_M_V))
    portfolio_allocations_W.append(np.array(portfolio_allocation_W_V))
    portfolio_allocations_SW.append(np.array(portfolio_allocation_SW_V))
    portfolio_allocations_MMSW_S0.append(np.array(portfolio_allocation_MMSW_V_S0))
    portfolio_allocations_MMSW_S1.append(np.array(portfolio_allocation_MMSW_V_S1))

# Convert lists to numpy arrays for easier manipulation
#portfolio_allocations_SW = np.array(portfolio_allocations_SW)
#portfolio_allocations_MMSW_S0 = np.array(portfolio_allocations_MMSW_S0)
#portfolio_allocations_MMSW_S1 = np.array(portfolio_allocations_MMSW_S1)

# Create Plotly graphs to visualize the results
fig = go.Figure()

# Add traces for both components of the portfolio allocation under Merton
#fig.add_trace(go.Scatter(x=lambda_1_space, y=np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='M Portfolio Allocation 1'))
fig.add_trace(go.Scatter(x=lambda_1_space, y=np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='M Portfolio Allocation 2'))

# Add traces for both components of the portfolio allocation under Wishart
#fig.add_trace(go.Scatter(x=lambda_1_space, y=np.squeeze(portfolio_allocations_W)[:, 0], mode='lines', name='W Portfolio Allocation 1'))
fig.add_trace(go.Scatter(x=lambda_1_space, y=np.squeeze(portfolio_allocations_W)[:, 1], mode='lines', name='W Portfolio Allocation 2'))

# Add traces for both components of the portfolio allocation under Shifted Wishart
#fig.add_trace(go.Scatter(x=delta_space, y=np.squeeze(portfolio_allocations_SW)[:, 0], mode='lines', name='SW Portfolio Allocation 1'))
#fig.add_trace(go.Scatter(x=delta_space, y=np.squeeze(portfolio_allocations_SW)[:, 1], mode='lines', name='SW Portfolio Allocation 2'))

# Add traces for both components of the portfolio allocation under MMSW State 0
#fig.add_trace(go.Scatter(x=lambda_1_space, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 0], mode='lines', name='MMSW Normal State Portfolio Allocation 1', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=lambda_1_space, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 1], mode='lines', name='MMSW Normal State Portfolio Allocation 2', line=dict(dash='dash')))

# Add traces for both components of the portfolio allocation under MMSW State 1
#fig.add_trace(go.Scatter(x=lambda_1_space, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 0], mode='lines', name='MMSW Crisis State Portfolio Allocation 1', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=lambda_1_space, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 1], mode='lines', name='MMSW Crisis State Portfolio Allocation 2', line=dict(dash='dash')))

# Customize the layout
fig.update_layout(
    #title="Portfolio Allocations under Merton, Wishart, Shifted Wishart and MMSW Processes",
    title="Sensitivity Analysis wrt Markov Modulation Parameters",
    xaxis_title="Markov Modulation parameter: $\lambda_1$",
    yaxis_title="Portfolio Allocation",
    legend_title="Portfolio Components",
)

fig.write_html('portfolio_allocations_shifted_wishart_mmsw.html')

# Display the figure
fig.show()


# Sensivity wrt p

In [ ]:
# Sensitivity wrt
# Define time horizon range t for tests
#time_horizon = np.linspace(0, 5, 100)
#lambda_0_space = np.linspace(0.2* lambda_bar_0, 5* lambda_bar_0, 100) # Sensitivity wrt lambda_0
p_space = np.linspace(0.1, 0.9, 100) # Sensitivity wrt lambda_1

tau= 5
# Compute portfolio allocations for both Theorems across the time horizon
portfolio_allocations_M = []
portfolio_allocations_W = []
portfolio_allocations_SW = []
portfolio_allocations_MMSW_S0 = []
portfolio_allocations_MMSW_S1 = []

#for lambda_0 in lambda_0_space: # Sensitivity wrt lambda_0
#for lambda_1 in lambda_1_space: # Sensitivity wrt lambda_1
for p in p_space:
    # Compute the block matrix F_ij(t) = exp( t * A )
    #A = np.block([[Gamma, -2 * Lambda], [C, -Gamma.T]])
    #F = expm(t * A)

    # Extract F_11, F_12, F_21, F_22 from F
    #F_21 = F[2:, :2]
    #F_22 = F[2:, 2:]

    # Compute B(t)
    #B_t = np.linalg.inv(F_22) @ F_21
    # Merton Portfolio Strategy under Black Scholes Process
    portfolio_allocation_M_V= portfolio_allocation_M(lambda_vector_M, delta)
    #portfolio_allocation_M_V= portfolio_allocation_M(lambda_vector_B, delta)

    # Buratchi Portfolio Strategy under Wishart Process
    portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_W, M_W,Q_W,rho_bar_W,tau, delta)
    #portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_B, M_B,Q_B,rho_bar_B,tau, delta)

    # Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
    #inv_Sigma_SW = np.linalg.inv(Sigma_SW)
    portfolio_allocation_SW_V = portfolio_allocation_SW(lambda_vector_SW, M_SW,Q_SW,rho_bar_SW,Sigma_inf_SW,sighat_SW,tau, delta)
    #portfolio_allocation_SW_V = portfolio_allocation_SW(lambda_vector_B, M_BS,Q_BS,rho_bar_BS,Sigma_inf_BS,sighat_BS, tau, delta)

    # Theorem 2: Optimal Portfolio Strategy under MMSW Process
    # Simulating a Markov chain that switches between states 0 and 1
    #state = np.random.choice([0, 1])  # Simulating the current state of the Markov chain
    #state = np.random.choice(2, p=transition_matrix[current_state])
    # The state-dependent widehat_Sigma
    #widehat_Sigma_MC = widehat_Sigma if state == 1 else np.zeros_like(widehat_Sigma)

    #sighat_MW= ((lambda_0 + lambda_bar_1)/lambda_bar_1)* sighat_N # Sensitivity wrt lambda_0
    sighat_MW= (1/(1-p))* sighat_N # Sensitivity wrt p
    # Use the state-dependent widehat_Sigma_MC in the MMSW process
    #inv_Sigma_MW = np.linalg.inv(Sigma_MW)
    # MW portfolio for normal state
    portfolio_allocation_MMSW_V_S0 = portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, 0, tau, delta)
    #portfolio_allocation_MMSW_V_S0 = portfolio_allocation_MMSW(lambda_vector_B, M_BM, Q_BM, rho_bar_BM, Sigma_inf_BM,sighat_BM, 0, tau, delta)
    # MW portfolio for Shifted State
    portfolio_allocation_MMSW_V_S1 = portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, 1, tau, delta)
    #portfolio_allocation_MMSW_V_S1 = portfolio_allocation_MMSW(lambda_vector_B, M_BM, Q_BM, rho_bar_BM, Sigma_inf_BM,sighat_BM, 1, tau, delta)

    # Store the results
    portfolio_allocations_M.append(np.array(portfolio_allocation_M_V))
    portfolio_allocations_W.append(np.array(portfolio_allocation_W_V))
    portfolio_allocations_SW.append(np.array(portfolio_allocation_SW_V))
    portfolio_allocations_MMSW_S0.append(np.array(portfolio_allocation_MMSW_V_S0))
    portfolio_allocations_MMSW_S1.append(np.array(portfolio_allocation_MMSW_V_S1))

# Convert lists to numpy arrays for easier manipulation
#portfolio_allocations_SW = np.array(portfolio_allocations_SW)
#portfolio_allocations_MMSW_S0 = np.array(portfolio_allocations_MMSW_S0)
#portfolio_allocations_MMSW_S1 = np.array(portfolio_allocations_MMSW_S1)

# Create Plotly graphs to visualize the results
fig = go.Figure()

# Add traces for both components of the portfolio allocation under Merton
fig.add_trace(go.Scatter(x=p_space, y=np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='Merton'))
#fig.add_trace(go.Scatter(x=p_space, y=np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='Merton'))

# Add traces for both components of the portfolio allocation under Wishart
fig.add_trace(go.Scatter(x=p_space, y=np.squeeze(portfolio_allocations_W)[:, 0], mode='lines', name='Wishart'))
#fig.add_trace(go.Scatter(x=p_space, y=np.squeeze(portfolio_allocations_W)[:, 1], mode='lines', name='Wishart'))

# Add traces for both components of the portfolio allocation under Shifted Wishart
fig.add_trace(go.Scatter(x=p_space, y=np.squeeze(portfolio_allocations_SW)[:, 0], mode='lines', name='Shifted W'))
#fig.add_trace(go.Scatter(x=p_space, y=np.squeeze(portfolio_allocations_SW)[:, 1], mode='lines', name='Shifted W'))

# Add traces for both components of the portfolio allocation under MMSW State 0
fig.add_trace(go.Scatter(x=p_space, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 0], mode='lines', name='MMSW Normal', line=dict(dash='dash')))
#fig.add_trace(go.Scatter(x=p_space, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 1], mode='lines', name='MMSW Normal', line=dict(dash='dash')))

# Add traces for both components of the portfolio allocation under MMSW State 1
fig.add_trace(go.Scatter(x=p_space, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 0], mode='lines', name='MMSW Crisis', line=dict(dash='dash')))
#fig.add_trace(go.Scatter(x=p_space, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 1], mode='lines', name='MMSW Crisis', line=dict(dash='dash')))

# Customize the layout
fig.update_layout(
#    title="Portfolio Allocations under Wishart Process",
    xaxis=dict(
        title="p",
        titlefont=dict(size=30),    # Font size of x-axis label
        tickfont=dict(size=30)      # Font size of x-axis tick numbers
    ),
    yaxis=dict(
        title="Portfolio Allocation",
        titlefont=dict(size=30),    # Font size of y-axis label
        tickfont=dict(size=30)      # Font size of y-axis tick numbers
    ),
    legend_title="Components",
    font=dict(size=30)  # General font (e.g., legend)
)


fig.write_html('portfolio_allocations_shifted_wishart_mmsw.html')

# Display the figure
fig.show()


# Sensitivity wrt $\delta$ and $\gamma$

In [ ]:
# This cell Marcos
# Define time horizon range t for tests
#time_horizon = np.linspace(0, 5, 100)
#delta_space = np.linspace(-10, -0.5, 100)
gamma_space=np.linspace(1.01, 20, 100)
tau= 5
# Compute portfolio allocations for both Theorems across the time horizon
portfolio_allocations_M = []
portfolio_allocations_W = []
portfolio_allocations_SW = []
portfolio_allocations_MMSW_S0 = []
portfolio_allocations_MMSW_S1 = []

#for delta in delta_space:
for gamma in gamma_space:
    delta= 1-gamma
    # Compute the block matrix F_ij(t) = exp( t * A )
    #A = np.block([[Gamma, -2 * Lambda], [C, -Gamma.T]])
    #F = expm(t * A)

    # Extract F_11, F_12, F_21, F_22 from F
    #F_21 = F[2:, :2]
    #F_22 = F[2:, 2:]

    # Compute B(t)
    #B_t = np.linalg.inv(F_22) @ F_21
    # Merton Portfolio Strategy under Black Scholes Process
    portfolio_allocation_M_V= portfolio_allocation_M(lambda_vector_M, delta)
    #portfolio_allocation_M_V= portfolio_allocation_M(lambda_vector_B, delta)

    # Buratchi Portfolio Strategy under Wishart Process
    portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_W, M_W,Q_W,rho_bar_W,tau, delta)
    #portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_B, M_B,Q_B,rho_bar_B,tau, delta)

    # Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
    #inv_Sigma_SW = np.linalg.inv(Sigma_SW)
    portfolio_allocation_SW_V = portfolio_allocation_SW(lambda_vector_SW, M_SW,Q_SW,rho_bar_SW,Sigma_inf_SW,sighat_SW,tau, delta)
    #portfolio_allocation_SW_V = portfolio_allocation_SW(lambda_vector_B, M_BS,Q_BS,rho_bar_BS,Sigma_inf_BS,sighat_BS, tau, delta)

    # Theorem 2: Optimal Portfolio Strategy under MMSW Process
    # Simulating a Markov chain that switches between states 0 and 1
    #state = np.random.choice([0, 1])  # Simulating the current state of the Markov chain
    #state = np.random.choice(2, p=transition_matrix[current_state])
    # The state-dependent widehat_Sigma
    #widehat_Sigma_MC = widehat_Sigma if state == 1 else np.zeros_like(widehat_Sigma)

    # Use the state-dependent widehat_Sigma_MC in the MMSW process
    #inv_Sigma_MW = np.linalg.inv(Sigma_MW)
    # MW portfolio for normal state
    portfolio_allocation_MMSW_V_S0 = portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, 0, tau, delta)
    #portfolio_allocation_MMSW_V_S0 = portfolio_allocation_MMSW(lambda_vector_B, M_BM, Q_BM, rho_bar_BM, Sigma_inf_BM,sighat_BM, 0, tau, delta)
    # MW portfolio for Shifted State
    portfolio_allocation_MMSW_V_S1 = portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, 1, tau, delta)
    #portfolio_allocation_MMSW_V_S1 = portfolio_allocation_MMSW(lambda_vector_B, M_BM, Q_BM, rho_bar_BM, Sigma_inf_BM,sighat_BM, 1, tau, delta)

    # Store the results
    portfolio_allocations_M.append(np.array(portfolio_allocation_M_V))
    portfolio_allocations_W.append(np.array(portfolio_allocation_W_V))
    portfolio_allocations_SW.append(np.array(portfolio_allocation_SW_V))
    portfolio_allocations_MMSW_S0.append(np.array(portfolio_allocation_MMSW_V_S0))
    portfolio_allocations_MMSW_S1.append(np.array(portfolio_allocation_MMSW_V_S1))

# Convert lists to numpy arrays for easier manipulation
#portfolio_allocations_SW = np.array(portfolio_allocations_SW)
#portfolio_allocations_MMSW_S0 = np.array(portfolio_allocations_MMSW_S0)
#portfolio_allocations_MMSW_S1 = np.array(portfolio_allocations_MMSW_S1)

# Create Plotly graphs to visualize the results
fig = go.Figure()

# Sensitivity wrt delta

# Add traces for both components of the portfolio allocation under Merton
#fig.add_trace(go.Scatter(x=delta_space, y=np.squeeze(portfolio_allocations_M)[:, 0]/np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='M Portfolio Allocation 1'))
#fig.add_trace(go.Scatter(x=delta_space, y=np.squeeze(portfolio_allocations_M)[:, 1]/np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='M Portfolio Allocation 2'))

# Add traces for both components of the portfolio allocation under Wishart
#fig.add_trace(go.Scatter(x=delta_space, y=np.squeeze(portfolio_allocations_W)[:, 0]/np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='W Portfolio Allocation 1'))
#fig.add_trace(go.Scatter(x=delta_space, y=np.squeeze(portfolio_allocations_W)[:, 1]/np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='W Portfolio Allocation 2'))

# Add traces for both components of the portfolio allocation under Shifted Wishart
#fig.add_trace(go.Scatter(x=delta_space, y=np.squeeze(portfolio_allocations_SW)[:, 0]/np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='SW Portfolio Allocation 1'))
#fig.add_trace(go.Scatter(x=delta_space, y=np.squeeze(portfolio_allocations_SW)[:, 1]/np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='SW Portfolio Allocation 2'))

# Add traces for both components of the portfolio allocation under MMSW State 0
#fig.add_trace(go.Scatter(x=delta_space, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 0]/np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='MMSW Normal State Portfolio Allocation 1', line=dict(dash='dash')))
#fig.add_trace(go.Scatter(x=delta_space, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 1]/np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='MMSW Normal State Portfolio Allocation 2', line=dict(dash='dash')))

# Add traces for both components of the portfolio allocation under MMSW State 1
#fig.add_trace(go.Scatter(x=delta_space, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 0]/np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='MMSW Crisis State Portfolio Allocation 1', line=dict(dash='dash')))
#fig.add_trace(go.Scatter(x=delta_space, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 1]/np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='MMSW Crisis State Portfolio Allocation 2', line=dict(dash='dash')))

# Sensitivity wrt gamma

# Add traces for both components of the portfolio allocation under Merton
fig.add_trace(go.Scatter(x=gamma_space, y=np.squeeze(portfolio_allocations_M)[:, 0]/np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='Merton'))
#fig.add_trace(go.Scatter(x=gamma_space, y=np.squeeze(portfolio_allocations_M)[:, 1]/np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='Merton'))

# Add traces for both components of the portfolio allocation under Wishart
fig.add_trace(go.Scatter(x=gamma_space, y=np.squeeze(portfolio_allocations_W)[:, 0]/np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='Wishart'))
#fig.add_trace(go.Scatter(x=gamma_space, y=np.squeeze(portfolio_allocations_W)[:, 1]/np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='Wishart'))

# Add traces for both components of the portfolio allocation under Shifted Wishart
fig.add_trace(go.Scatter(x=gamma_space, y=np.squeeze(portfolio_allocations_SW)[:, 0]/np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='Shifted W'))
#fig.add_trace(go.Scatter(x=gamma_space, y=np.squeeze(portfolio_allocations_SW)[:, 1]/np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='Shifted W'))

# Add traces for both components of the portfolio allocation under MMSW State 0
fig.add_trace(go.Scatter(x=gamma_space, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 0]/np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='MMSW Normal', line=dict(dash='dash')))
#fig.add_trace(go.Scatter(x=gamma_space, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 1]/np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='MMSW Normal', line=dict(dash='dash')))

# Add traces for both components of the portfolio allocation under MMSW State 1
fig.add_trace(go.Scatter(x=gamma_space, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 0]/np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='MMSW Crisis', line=dict(dash='dash')))
#fig.add_trace(go.Scatter(x=gamma_space, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 1]/np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='MMSW Crisis', line=dict(dash='dash')))

# Customize the layout

fig.update_layout(
#    title="Portfolio Allocations under Wishart Process",
    xaxis=dict(
        title="Risk Aversion",
        titlefont=dict(size=30),    # Font size of x-axis label
        tickfont=dict(size=30)      # Font size of x-axis tick numbers
    ),
    yaxis=dict(
        title="Portfolio Ratio",
        titlefont=dict(size=30),    # Font size of y-axis label
        tickfont=dict(size=30)      # Font size of y-axis tick numbers
    ),
    legend_title="Components",
    font=dict(size=30)  # General font (e.g., legend)
)


fig.write_html('portfolio_allocations_shifted_wishart_mmsw.html')

# Display the figure
fig.show()


# Sensitivity for $\widehat{\Sigma}$

In [ ]:
U, S, Vt = np.linalg.svd(Sigma_inf_SW)
Sig_hat_= U @ np.diag(np.array([S[0],0])) @ Vt

In [ ]:
M= U @ np.diag(np.array([S[0],0])) @ Vt

In [ ]:
# Define time horizon range t for tests
#time_horizon = np.linspace(0, 5, 100)
theta_space = np.linspace(0, 2, 100)

tau= 5
# Compute portfolio allocations for both Theorems across the time horizon
portfolio_allocations_M = []
portfolio_allocations_W = []
portfolio_allocations_SW = []
portfolio_allocations_MMSW_S0 = []
portfolio_allocations_MMSW_S1 = []


for theta in theta_space:
    # Compute the block matrix F_ij(t) = exp( t * A )
    #A = np.block([[Gamma, -2 * Lambda], [C, -Gamma.T]])
    #F = expm(t * A)

    # Extract F_11, F_12, F_21, F_22 from F
    #F_21 = F[2:, :2]
    #F_22 = F[2:, 2:]

    # Compute B(t)
    #B_t = np.linalg.inv(F_22) @ F_21
    # Merton Portfolio Strategy under Black Scholes Process
    portfolio_allocation_M_V= portfolio_allocation_M(lambda_vector_M, delta)
    #portfolio_allocation_M_V= portfolio_allocation_M(lambda_vector_B, delta)

    # Buratchi Portfolio Strategy under Wishart Process
    portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_W, M_W,Q_W,rho_bar_W,tau, delta)
    #portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_B, M_B,Q_B,rho_bar_B,tau, delta)

    #sighat_SW= theta * sighat_T
    #sighat_SW= theta * Sigma_inf_SW
    #Sigma_inf_SW_off_diag= np.array([[0, Sigma_inf_SW[0,1]], [Sigma_inf_SW[1,0], 0]]) # Sensitivity Analysis wrt to the Correlation
    #Sigma_inf_SW_off_diag= np.array([[Sigma_inf_SW[0,0], 0], [0, Sigma_inf_SW[1,1]]]) # Sensitivity Analysis wrt to the Variance & Correlation
    #Sigma_inf_SW_off_diag= np.asmatrix(Sigma_inf_SW_off_diag)
    #sighat_SW= theta * Sigma_inf_SW_off_diag
    # Perform SVD
    U, S, Vt = np.linalg.svd(Sigma_inf_SW) # wrt to the eigenvalue
    Sig_hat_= U @ np.diag(np.array([S[0],0])) @ Vt
    sighat_SW= theta * Sig_hat_
    # Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
    #inv_Sigma_SW = np.linalg.inv(Sigma_SW)
    portfolio_allocation_SW_V = portfolio_allocation_SW(lambda_vector_SW, M_SW,Q_SW,rho_bar_SW,Sigma_inf_SW,sighat_SW,tau, delta)
    #portfolio_allocation_SW_V = portfolio_allocation_SW(lambda_vector_B, M_BS,Q_BS,rho_bar_BS,Sigma_inf_BS,sighat_BS, tau, delta)

    # Theorem 2: Optimal Portfolio Strategy under MMSW Process
    # Simulating a Markov chain that switches between states 0 and 1
    #state = np.random.choice([0, 1])  # Simulating the current state of the Markov chain
    #state = np.random.choice(2, p=transition_matrix[current_state])
    # The state-dependent widehat_Sigma
    #widehat_Sigma_MC = widehat_Sigma if state == 1 else np.zeros_like(widehat_Sigma)


    #sighat_MW= theta * ((lambda_bar_0 + lambda_bar_1)/lambda_bar_1)* sighat_N
    #sighat_MW= theta * Sigma_inf_MW
    # Sigma_inf_MW_off_diag= np.array([[0, Sigma_inf_MW[0,1]], [Sigma_inf_MW[1,0], 0]]) # Sensitivity Analysis wrt to the Correlation
    #Sigma_inf_MW_off_diag= np.array([[Sigma_inf_MW[0,0], 0], [0, Sigma_inf_MW[1,1]]]) # Sensitivity Analysis wrt to the Variance & Correlation
    #Sigma_inf_MW_off_diag= np.asmatrix(Sigma_inf_MW_off_diag)
    #sighat_MW= theta * Sigma_inf_MW_off_diag
    U, S, Vt = np.linalg.svd(Sigma_inf_MW) # wrt to the eigenvalue
    Sig_hat_= U @ np.diag(np.array([S[0],0])) @ Vt
    sighat_MW= theta * Sig_hat_
    # Use the state-dependent widehat_Sigma_MC in the MMSW process
    #inv_Sigma_MW = np.linalg.inv(Sigma_MW)
    # MW portfolio for normal state
    portfolio_allocation_MMSW_V_S0 = portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, 0, tau, delta)
    #portfolio_allocation_MMSW_V_S0 = portfolio_allocation_MMSW(lambda_vector_B, M_BM, Q_BM, rho_bar_BM, Sigma_inf_BM,sighat_BM, 0, tau, delta)
    # MW portfolio for Shifted State
    portfolio_allocation_MMSW_V_S1 = portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, 1, tau, delta)
    #portfolio_allocation_MMSW_V_S1 = portfolio_allocation_MMSW(lambda_vector_B, M_BM, Q_BM, rho_bar_BM, Sigma_inf_BM,sighat_BM, 1, tau, delta)

    # Store the results
    portfolio_allocations_M.append(np.array(portfolio_allocation_M_V))
    portfolio_allocations_W.append(np.array(portfolio_allocation_W_V))
    portfolio_allocations_SW.append(np.array(portfolio_allocation_SW_V))
    portfolio_allocations_MMSW_S0.append(np.array(portfolio_allocation_MMSW_V_S0))
    portfolio_allocations_MMSW_S1.append(np.array(portfolio_allocation_MMSW_V_S1))

# Convert lists to numpy arrays for easier manipulation
#portfolio_allocations_SW = np.array(portfolio_allocations_SW)
#portfolio_allocations_MMSW_S0 = np.array(portfolio_allocations_MMSW_S0)
#portfolio_allocations_MMSW_S1 = np.array(portfolio_allocations_MMSW_S1)

# Create Plotly graphs to visualize the results
fig = go.Figure()

# Add traces for both components of the portfolio allocation under Merton
#fig.add_trace(go.Scatter(x=theta_space, y=np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='M Portfolio Allocation 1'))
fig.add_trace(go.Scatter(x=theta_space, y=np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='M Portfolio Allocation 2'))

# Add traces for both components of the portfolio allocation under Wishart
#fig.add_trace(go.Scatter(x=theta_space, y=np.squeeze(portfolio_allocations_W)[:, 0], mode='lines', name='W Portfolio Allocation 1'))
fig.add_trace(go.Scatter(x=theta_space, y=np.squeeze(portfolio_allocations_W)[:, 1], mode='lines', name='W Portfolio Allocation 2'))

# Add traces for both components of the portfolio allocation under Shifted Wishart
#fig.add_trace(go.Scatter(x=theta_space, y=np.squeeze(portfolio_allocations_SW)[:, 0], mode='lines', name='SW Portfolio Allocation 1'))
fig.add_trace(go.Scatter(x=theta_space, y=np.squeeze(portfolio_allocations_SW)[:, 1], mode='lines', name='SW Portfolio Allocation 2'))

# Add traces for both components of the portfolio allocation under MMSW State 0
#fig.add_trace(go.Scatter(x=theta_space, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 0], mode='lines', name='MMSW Normal State Portfolio Allocation 1', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=theta_space, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 1], mode='lines', name='MMSW Normal State Portfolio Allocation 2', line=dict(dash='dash')))

# Add traces for both components of the portfolio allocation under MMSW State 1
#fig.add_trace(go.Scatter(x=theta_space, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 0], mode='lines', name='MMSW Crisis State Portfolio Allocation 1', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=theta_space, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 1], mode='lines', name='MMSW Crisis State Portfolio Allocation 2', line=dict(dash='dash')))

# Customize the layout
fig.update_layout(
    #title="Portfolio Allocations under Merton, Wishart, Shifted Wishart and MMSW Processes",
    title="Sensitivity Analysis wrt the first Eigenvalue",
    xaxis_title="Shift Factor: $\eta$",
    yaxis_title="Portfolio Allocation",
    legend_title="Portfolio Components",
)

fig.write_html('portfolio_allocations_shifted_wishart_mmsw.html')

# Display the figure
fig.show()


# Sensitivity Analysis wrt $\widehat{\Sigma}$

In [ ]:
Sigma_inf_max= np.array([[0.64, 0.576], [0.576, 0.64]]) # corresponding to max std=0.8 and corr=0.9
sighat_hat_inf_MW= Sigma_inf_max - Sigma_inf_normal_N
#sighat_MW= 1/(1-p) * sighat_hat_inf_MW
sighat_MW=  1 * sighat_hat_inf_MW

In [ ]:
sighat_MW

array([[0.6192, 0.5528],
       [0.5528, 0.5386]])

In [ ]:
Sigma_inf_normal_N + sighat_MW

array([[0.64 , 0.576],
       [0.576, 0.64 ]])

In [ ]:
tau=5
print("estimated p, estimated sighat_MW")
print("sighat_MW", sighat_MW)

print("Merton: Optimal Portfolio Strategy under BS Process")
print(portfolio_allocation_M(lambda_vector_M, delta))

print("Buraschi: Optimal Portfolio Strategy under Wishart Process")
print(portfolio_allocation_W(lambda_vector_W, M_W,Q_W,rho_bar_W,tau, delta))

print("Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process")
print(portfolio_allocation_SW(lambda_vector_SW, M_SW,Q_SW,rho_bar_SW,Sigma_inf_SW,sighat_SW,tau, delta))

state= 0
print("\nTheorem 2: Optimal Portfolio Strategy under MMSW Process: State 0")
print(portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, state, tau, delta))

state= 1
print("\nTheorem 2: Optimal Portfolio Strategy under MMSW Process: State 1")
print(portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, state, tau, delta))

estimated p, estimated sighat_MW
sighat_MW [[0.07343235 0.07615207]
 [0.07615207 0.04215561]]
Merton: Optimal Portfolio Strategy under BS Process
[[-0.43985937]
 [ 0.57311992]]
Buraschi: Optimal Portfolio Strategy under Wishart Process
[[-0.41701709]
 [ 0.60232078]]
Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
[[-0.41409103]
 [ 0.59691069]]

Theorem 2: Optimal Portfolio Strategy under MMSW Process: State 0
[[-0.41455011]
 [ 0.60242794]]

Theorem 2: Optimal Portfolio Strategy under MMSW Process: State 1
[[-0.54528115]
 [ 0.62109454]]


# Sensitivity Analysis wrt $\Sigma_∞^{MW}$




In [ ]:
Sigma_inf_max= np.array([[0.64, 0.576], [0.576, 0.64]])


In [ ]:
Sigma_inf_MW= 1 * Sigma_inf_max

In [ ]:
#Sigma_inf_MW= np.array([[0.32, 0.2], [0.2, 0.32]])
Sigma_inf_MW= np.array([[0.1, 0.1], [0.1, 0.2]])

In [ ]:
tau=5
print("")
print("Sigma_inf_MW", Sigma_inf_MW)
print("Correlation: ", Sigma_inf_MW[0,1] / (np.sqrt(Sigma_inf_MW[0,0] * Sigma_inf_MW[1,1])))

print("Merton: Optimal Portfolio Strategy under BS Process")
print(portfolio_allocation_M(lambda_vector_M, delta))

print("Buraschi: Optimal Portfolio Strategy under Wishart Process")
print(portfolio_allocation_W(lambda_vector_W, M_W,Q_W,rho_bar_W,tau, delta))

print("Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process")
print(portfolio_allocation_SW(lambda_vector_SW, M_SW,Q_SW,rho_bar_SW,Sigma_inf_SW,sighat_SW,tau, delta))

state= 0
print("\nTheorem 2: Optimal Portfolio Strategy under MMSW Process: State 0")
print(portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, state, tau, delta))

state= 1
print("\nTheorem 2: Optimal Portfolio Strategy under MMSW Process: State 1")
print(portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, state, tau, delta))


Sigma_inf_MW [[0.1 0.1]
 [0.1 0.2]]
Correlation:  0.7071067811865475
Merton: Optimal Portfolio Strategy under BS Process
[[-0.43985937]
 [ 0.57311992]]
Buraschi: Optimal Portfolio Strategy under Wishart Process
[[-0.41701709]
 [ 0.60232078]]
Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
[[-0.41409103]
 [ 0.59691069]]

Theorem 2: Optimal Portfolio Strategy under MMSW Process: State 0
[[-0.41455011]
 [ 0.60242794]]

Theorem 2: Optimal Portfolio Strategy under MMSW Process: State 1
[[-0.46472931]
 [ 0.61170329]]


## Varying the correlation vector

In [ ]:
Sigma_inf_M

array([[0.2479, 0.0836],
       [0.0836, 0.1023]])

In [ ]:
Sigma_inf_W

array([[0.1472, 0.0589],
       [0.0589, 0.0692]])

In [ ]:
# Define a range of correlation values to test
correlation_values = np.linspace(0, 0.9, 100)

# Arrays to store the portfolio allocations for both processes
portfolio_allocations_SW_correlation = []
portfolio_allocations_MMSW_correlation = []

# Perform the test by varying the correlation vector (same correlation for all dimensions for simplicity)
for corr in correlation_values:
    # Update the correlation vector
    rho_bar_test = np.array([corr, corr])

    # Use a fixed time point for comparison
    t = 1.0  # Fixed time for the test

    # Compute the block matrix F_ij(t) = exp( t * A )
    A = np.block([[Gamma, -2 * Lambda], [C, -Gamma.T]])
    F = expm(t * A)

    # Extract F_11, F_12, F_21, F_22 from F
    F_21 = F[2:, :2]
    F_22 = F[2:, 2:]

    # Compute B(t)
    B_t = np.linalg.inv(F_22) @ F_21

    # Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
    inv_Sigma_SW = np.linalg.inv(Sigma_SW)
    portfolio_allocation_SW = (1 / (1 - delta)) * lambda_vector + 2 * B_t @ Q.T @ rho_bar_test - 2 * inv_Sigma_SW @ widehat_Sigma @ B_t @ Q.T @ rho_bar_test

    # Theorem 2: Optimal Portfolio Strategy under MMSW Process
    # Simulating a Markov chain that switches between states 0 and 1
    state = np.random.choice([0, 1])  # Simulating the current state of the Markov chain

    # The state-dependent widehat_Sigma
    widehat_Sigma_MC = widehat_Sigma if state == 1 else np.zeros_like(widehat_Sigma)

    # Use the state-dependent widehat_Sigma_MC in the MMSW process
    inv_Sigma_MW = np.linalg.inv(Sigma_MW)
    portfolio_allocation_MMSW = (1 / (1 - delta)) * lambda_vector + 2 * B_t @ Q.T @ rho_bar_test - 2 * inv_Sigma_MW @ widehat_Sigma_MC @ B_t @ Q.T @ rho_bar_test

    # Store the results
    portfolio_allocations_SW_correlation.append(portfolio_allocation_SW)
    portfolio_allocations_MMSW_correlation.append(portfolio_allocation_MMSW)

# Convert lists to numpy arrays for easier manipulation
portfolio_allocations_SW_correlation = np.array(portfolio_allocations_SW_correlation)
portfolio_allocations_MMSW_correlation = np.array(portfolio_allocations_MMSW_correlation)

# Create Plotly graphs to visualize the results
fig = go.Figure()

# Add traces for both components of the portfolio allocation under Shifted Wishart for varying correlation
fig.add_trace(go.Scatter(x=correlation_values, y=portfolio_allocations_SW_correlation[:, 0], mode='lines', name='SW Portfolio Allocation 1'))
fig.add_trace(go.Scatter(x=correlation_values, y=portfolio_allocations_SW_correlation[:, 1], mode='lines', name='SW Portfolio Allocation 2'))

# Add traces for both components of the portfolio allocation under MMSW for varying correlation
fig.add_trace(go.Scatter(x=correlation_values, y=portfolio_allocations_MMSW_correlation[:, 0], mode='lines', name='MMSW Portfolio Allocation 1', line=dict(dash='dash')))
fig.add_trace(go.Scatter(x=correlation_values, y=portfolio_allocations_MMSW_correlation[:, 1], mode='lines', name='MMSW Portfolio Allocation 2', line=dict(dash='dash')))

# Customize the layout
fig.update_layout(
    title="Portfolio Allocations under Shifted Wishart and MMSW Processes for Different Correlations",
    xaxis_title="Correlation (\u03C1)",
    yaxis_title="Portfolio Allocation",
    legend_title="Portfolio Components",
)

fig.show()

# Save the figure to a file and display
fig.write_html('portfolio_allocations_correlation_shifted_wishart_mmsw.html')


## Simulation of portfolio value growth for SW and MMSW processes


In [ ]:
# Define initial portfolio value and time horizon
initial_value = 1000  # Initial portfolio value
time_steps = 100  # Number of time steps for the simulation
time_horizon = np.linspace(0, 2, time_steps)  # Time horizon from 0 to 2 years
dt = time_horizon[1] - time_horizon[0]  # Time step size

# Arrays to store portfolio values over time for both strategies
portfolio_values_SW = [initial_value]
portfolio_values_MMSW = [initial_value]

# Simulate the portfolio value evolution over time
for t in range(1, time_steps):
    # Compute the block matrix F_ij(t) = exp( t * A )
    A = np.block([[Gamma, -2 * Lambda], [C, -Gamma.T]])
    F = expm(time_horizon[t] * A)

    # Extract F_11, F_12, F_21, F_22 from F
    F_21 = F[2:, :2]
    F_22 = F[2:, 2:]

    # Compute B(t)
    B_t = np.linalg.inv(F_22) @ F_21

    # Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
    inv_Sigma_SW = np.linalg.inv(Sigma_SW)
    portfolio_allocation_SW = (1 / (1 - delta)) * lambda_vector + 2 * B_t @ Q.T @ rho_bar - 2 * inv_Sigma_SW @ widehat_Sigma @ B_t @ Q.T @ rho_bar

    # Simulate the portfolio growth based on the allocation and drift (mu * dt)
    mu_SW = np.dot(portfolio_allocation_SW, lambda_vector)
    portfolio_value_SW = portfolio_values_SW[-1] * np.exp(mu_SW * dt)
    portfolio_values_SW.append(portfolio_value_SW)

    # Theorem 2: Optimal Portfolio Strategy under MMSW Process
    state = np.random.choice([0, 1])  # Simulating the current state of the Markov chain
    widehat_Sigma_MC = widehat_Sigma if state == 1 else np.zeros_like(widehat_Sigma)
    inv_Sigma_MW = np.linalg.inv(Sigma_MW)
    portfolio_allocation_MMSW = (1 / (1 - delta)) * lambda_vector + 2 * B_t @ Q.T @ rho_bar - 2 * inv_Sigma_MW @ widehat_Sigma_MC @ B_t @ Q.T @ rho_bar

    # Simulate the portfolio growth based on the allocation and drift (mu * dt)
    mu_MMSW = np.dot(portfolio_allocation_MMSW, lambda_vector)
    portfolio_value_MMSW = portfolio_values_MMSW[-1] * np.exp(mu_MMSW * dt)
    portfolio_values_MMSW.append(portfolio_value_MMSW)

# Convert portfolio values to numpy arrays for easier manipulation
portfolio_values_SW = np.array(portfolio_values_SW)
portfolio_values_MMSW = np.array(portfolio_values_MMSW)

# Create Plotly graphs to visualize the results
fig = go.Figure()

# Add traces for portfolio value under Shifted Wishart process
fig.add_trace(go.Scatter(x=time_horizon, y=portfolio_values_SW, mode='lines', name='SW Portfolio Value'))

# Add traces for portfolio value under MMSW process
fig.add_trace(go.Scatter(x=time_horizon, y=portfolio_values_MMSW, mode='lines', name='MMSW Portfolio Value', line=dict(dash='dash')))

# Customize the layout
fig.update_layout(
    title="Portfolio Value Growth under Shifted Wishart and MMSW Processes",
    xaxis_title="Time",
    yaxis_title="Portfolio Value",
    legend_title="Portfolio Strategy",
)

fig.show()

# Save the figure to a file and display
fig.write_html('portfolio_value_growth_sw_vs_mmsw.html')


## Portfolio value growth over volatility (sigma) and correlation

In [ ]:
# Define initial portfolio value, correlation range, and volatility range
initial_value = 1000  # Initial portfolio value
correlation_steps = 50  # Number of correlation steps for the simulation
volatility_steps = 50  # Number of volatility steps for the simulation
correlation_values = np.linspace(0.1, 0.9, correlation_steps)  # Range of correlation values
volatility_values = np.linspace(0.1, 0.5, volatility_steps)  # Range of volatility values

# Arrays to store portfolio values for both strategies across volatility and correlation
portfolio_values_SW_surface_volatility_corr = np.zeros((volatility_steps, correlation_steps))
portfolio_values_MMSW_surface_volatility_corr = np.zeros((volatility_steps, correlation_steps))

# Simulate the portfolio value evolution over volatility and correlation
for i, vol in enumerate(volatility_values):
    for j, corr in enumerate(correlation_values):
        # Update the volatility and correlation in the lambda vector and correlation vector
        lambda_vector_test = np.array([vol, vol])
        rho_bar_test = np.array([corr, corr])

        # Use a fixed time point for comparison
        t = 1.0  # Fixed time for the test

        # Compute the block matrix F_ij(t) = exp( t * A )
        A = np.block([[Gamma, -2 * Lambda], [C, -Gamma.T]])
        F = expm(t * A)

        # Extract F_11, F_12, F_21, F_22 from F
        F_21 = F[2:, :2]
        F_22 = F[2:, 2:]

        # Compute B(t)
        B_t = np.linalg.inv(F_22) @ F_21

        # Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
        inv_Sigma_SW = np.linalg.inv(Sigma_SW)
        portfolio_allocation_SW = (1 / (1 - delta)) * lambda_vector_test + 2 * B_t @ Q.T @ rho_bar_test - 2 * inv_Sigma_SW @ widehat_Sigma @ B_t @ Q.T @ rho_bar_test

        # Simulate the portfolio growth based on the allocation and drift (mu * dt)
        mu_SW = np.dot(portfolio_allocation_SW, lambda_vector_test)
        portfolio_value_SW = initial_value * np.exp(mu_SW * t)
        portfolio_values_SW_surface_volatility_corr[i, j] = portfolio_value_SW

        # Theorem 2: Optimal Portfolio Strategy under MMSW Process
        state = np.random.choice([0, 1])  # Simulating the current state of the Markov chain
        widehat_Sigma_MC = widehat_Sigma if state == 1 else np.zeros_like(widehat_Sigma)
        inv_Sigma_MW = np.linalg.inv(Sigma_MW)
        portfolio_allocation_MMSW = (1 / (1 - delta)) * lambda_vector_test + 2 * B_t @ Q.T @ rho_bar_test - 2 * inv_Sigma_MW @ widehat_Sigma_MC @ B_t @ Q.T @ rho_bar_test

        # Simulate the portfolio growth based on the allocation and drift (mu * dt)
        mu_MMSW = np.dot(portfolio_allocation_MMSW, lambda_vector_test)
        portfolio_value_MMSW = initial_value * np.exp(mu_MMSW * t)
        portfolio_values_MMSW_surface_volatility_corr[i, j] = portfolio_value_MMSW

# Create a surface plot using Plotly
fig = go.Figure()

# Add surface plot for Shifted Wishart process
fig.add_trace(go.Surface(z=portfolio_values_SW_surface_volatility_corr, x=correlation_values, y=volatility_values, colorscale='Viridis', name='SW Portfolio Value'))

# Add surface plot for MMSW process
fig.add_trace(go.Surface(z=portfolio_values_MMSW_surface_volatility_corr, x=correlation_values, y=volatility_values, colorscale='Cividis', name='MMSW Portfolio Value', showscale=False))

# Customize the layout
fig.update_layout(
    title="Portfolio Value Growth Surface under Shifted Wishart and MMSW Processes (Volatility and Correlation)",
    scene=dict(
        xaxis_title='Correlation (\u03C1)',
        yaxis_title='Volatility (\u03C3)',
        zaxis_title='Portfolio Value',
    ),
)

fig.show()

# Save the figure to a file and display
fig.write_html('portfolio_value_surface_volatility_correlation_sw_vs_mmsw.html')


## Covariance Simulation

In [ ]:
M_B= np.array([[-1.122,0.747], [0.884, -0.888]])
Q_B= np.array([[0.160, 0.083], [-0.021, 0.009]])
rho_bar_B= np.array([-0.279, -0.247])
Sigma_inf_B= np.array([[0.37418953, 0.3877318] ,[0.3877318, 0.42523076]])
beta_B=10


In [ ]:
import scipy
import scipy.stats as ss

#dt=1/250
dt=1/252
N=100000
paths=1


#parameters of Wishart
beta=beta_B
#Sigma_0= Sigma_inf_normal_T
Sigma_0= Sigma_inf_B
M=M_B
Q=Q_B

#Wishart Simulation using Euler-Maruyama Simulation for Buraschi parameters
Sigma_EM = np.zeros((N, paths, 2, 2))
Sigma_EM[0,:]= Sigma_0
W = ss.norm.rvs(loc=0, scale=1, size=(N - 1, paths, 2 , 2))
for i in range(0,N-1):
    for j in range(paths):
        sqrt_Sigma_EM= np.real(scipy.linalg.sqrtm(Sigma_EM[i,j]))
        drift=  (beta * (np.matrix.transpose(Q) @ Q) + M @ Sigma_EM[i,j] + Sigma_EM[i,j] @ np.matrix.transpose(M))* dt
        diffusion= np.sqrt(dt) * ( (sqrt_Sigma_EM @ W[i,j]) @ Q +  (np.matrix.transpose(Q) @ np.matrix.transpose(W[i,j])) @ sqrt_Sigma_EM)
        Sigma_EM[i+1,j] = Sigma_EM[i, j] + drift + diffusion


# Ensure all values are symmetric and positive semi-definite
for i in range(N):
    for j in range(paths):
        Sigma_EM[i, j] = (Sigma_EM[i, j] + np.transpose(Sigma_EM[i, j])) / 2  # Symmetrize to avoid numerical issues
        if np.any(np.linalg.eigvals(Sigma_EM[i, j]) <= 0):
            Sigma_EM[i, j] += -1.1 * np.min(np.linalg.eigvals(Sigma_EM[i, j])) * np.eye(2)  # Make positive definite

#Sigma_WP_B= Sigma_EM
Sigma_WP= Sigma_EM

"""
#parameters of Wishart
beta=beta_T
Sigma_0= Sigma_inf_total_T
M=M_T
Q=Q_T

#Wishart Simulation using Euler-Maruyama Simulation for SP-AAPL parameters
Sigma_EM = np.zeros((N, paths, 2, 2))
Sigma_EM[0,:]= Sigma_0
W = ss.norm.rvs(loc=0, scale=1, size=(N - 1, paths, 2 , 2))
for i in range(0,N-1):
    for j in range(paths):
        sqrt_Sigma_EM= np.real(scipy.linalg.sqrtm(Sigma_EM[i,j]))
        drift=  (beta * (np.matrix.transpose(Q) @ Q) + M @ Sigma_EM[i,j] + Sigma_EM[i,j] @ np.matrix.transpose(M))* dt
        diffusion= np.sqrt(dt) * ( (sqrt_Sigma_EM @ W[i,j]) @ Q +  (np.matrix.transpose(Q) @ np.matrix.transpose(W[i,j])) @ sqrt_Sigma_EM)
        Sigma_EM[i+1,j] = Sigma_EM[i, j] + drift + diffusion


# Ensure all values are symmetric and positive semi-definite
for i in range(N):
    for j in range(paths):
        Sigma_EM[i, j] = (Sigma_EM[i, j] + np.transpose(Sigma_EM[i, j])) / 2  # Symmetrize to avoid numerical issues
        if np.any(np.linalg.eigvals(Sigma_EM[i, j]) <= 0):
            Sigma_EM[i, j] += -1.1 * np.min(np.linalg.eigvals(Sigma_EM[i, j])) * np.eye(2)  # Make positive definite

Sigma_WP_SPAPL= Sigma_EM
#Sigma_WP= Sigma_EM
"""

'\n#parameters of Wishart\nbeta=beta_T\nSigma_0= Sigma_inf_total_T\nM=M_T\nQ=Q_T\n\n#Wishart Simulation using Euler-Maruyama Simulation for SP-AAPL parameters\nSigma_EM = np.zeros((N, paths, 2, 2))\nSigma_EM[0,:]= Sigma_0\nW = ss.norm.rvs(loc=0, scale=1, size=(N - 1, paths, 2 , 2))\nfor i in range(0,N-1):\n    for j in range(paths):\n        sqrt_Sigma_EM= np.real(scipy.linalg.sqrtm(Sigma_EM[i,j]))\n        drift=  (beta * (np.matrix.transpose(Q) @ Q) + M @ Sigma_EM[i,j] + Sigma_EM[i,j] @ np.matrix.transpose(M))* dt\n        diffusion= np.sqrt(dt) * ( (sqrt_Sigma_EM @ W[i,j]) @ Q +  (np.matrix.transpose(Q) @ np.matrix.transpose(W[i,j])) @ sqrt_Sigma_EM)\n        Sigma_EM[i+1,j] = Sigma_EM[i, j] + drift + diffusion\n\n\n# Ensure all values are symmetric and positive semi-definite\nfor i in range(N):\n    for j in range(paths):\n        Sigma_EM[i, j] = (Sigma_EM[i, j] + np.transpose(Sigma_EM[i, j])) / 2  # Symmetrize to avoid numerical issues\n        if np.any(np.linalg.eigvals(Sigma_E

In [ ]:
# Generating prices for Wishart Process
ln_P = np.zeros((N, paths, 2))
#y_0= np.array([6.9755,-1.1141])
y_0= np.array([1,1])
#rho= np.array([-0.4388, -0.2252])
#rho= np.array([0, 0])
rho= rho_bar_B
#mu= np.array([0.0983, 0.3700])
mu= np.array([0, 0])
ln_P[0,:]= y_0
#W_P= ss.norm.rvs(loc=0, scale=1, size=(N - 1, paths, 2 ))
#generating W_P with estimated correlations
dB= ss.norm.rvs(loc=0, scale=1, size=(N - 1, paths, 2 ))
W_P= np.sqrt(1-np.dot(rho, rho)) * dB + W[:,:,] @ rho
for path in range(paths):
    for t in range(N-1):
        # Here wo do matrix calculations (sqrtm, multiplication) for np.array objects as matrices, a point to check whether it works correctly
        drift= dt * (mu - 0.5 * np.diag(Sigma_WP[t,path]))
        # without - 0.5 * np.diag(Sigma_WP[t,path]) drift as in Da Fonseca model
        #drift= dt * (mu )
        diffusion= np.sqrt(dt) * np.matmul(np.real(scipy.linalg.sqrtm(Sigma_WP[t,path])) , W_P[t,path])
        #ln_P[t+1, path]= ln_P[t, path] + dt * (mu - 0.5 * np.diag(Sigma_WP[t,path])) + np.sqrt(dt) * np.matmul(scipy.linalg.sqrtm(Sigma_WP[t,path]) , W_P[t,path])
        ln_P[t+1, path]= ln_P[t, path] + drift + diffusion
P= np.exp(ln_P)


In [ ]:
# extracting time series for the prices
SP_sim_price= pd.Series(P[:,0].T[0])
APL_sim_price= pd.Series(P[:,0].T[1])
# Computing returns
SP_sim_ret= np.log(1+SP_sim_price.pct_change()).dropna()
APL_sim_ret= np.log(1+APL_sim_price.pct_change()).dropna()
# Computing the covariance
covariance_matrix= np.cov(SP_sim_ret, APL_sim_ret)

In [ ]:
252 * covariance_matrix

array([[0.37878479, 0.39283918],
       [0.39283918, 0.43107804]])

In [ ]:
Sigma_0

array([[0.0237, 0.0202],
       [0.0202, 0.0243]])

In [ ]:
# Create Plotly graphs to visualize the results
fig = go.Figure()
x_=np.linspace(0,1,N)
# Add traces for both components of the portfolio allocation under Merton
for i in range(paths):
  #
  fig.add_trace(go.Scatter(x=x_, y=Sigma_WP_B[:,i,1,1], mode='lines'))

  # Add traces for both components of the portfolio allocation under Wishart
  fig.add_trace(go.Scatter(x=x_, y=Sigma_WP_SPAPL[:,i,1,1], mode='lines'))  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_W)[:, 1], mode='lines', name='W Portfolio Allocation 2'))

  # Add traces for both components of the portfolio allocation under Shifted Wishart
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_SW)[:, 0], mode='lines', name='SW Portfolio Allocation 1'))
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_SW)[:, 1], mode='lines', name='SW Portfolio Allocation 2'))

  # Add traces for both components of the portfolio allocation under MMSW State 0
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 0], mode='lines', name='MMSW Normal State Portfolio Allocation 1', line=dict(dash='dash')))
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 1], mode='lines', name='MMSW Normal State Portfolio Allocation 2', line=dict(dash='dash')))

  # Add traces for both components of the portfolio allocation under MMSW State 1
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 0], mode='lines', name='MMSW Chrisis State Portfolio Allocation 1', line=dict(dash='dash')))
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 1], mode='lines', name='MMSW Chrisis State Portfolio Allocation 2', line=dict(dash='dash')))

# Customize the layout
fig.update_layout(
    title="50 paths for Wishart Covariance[1,1] from Buraschi vs SP-AAPL",
    xaxis_title="Time Horizon ",
    yaxis_title="Cov[1,1]",
    legend_title="",
)

fig.write_html('portfolio_allocations_shifted_wishart_mmsw.html')

# Display the figure
fig.show()


In [ ]:
# Wishart Process
rho_bar= np.array([-0.279, -0.247]) # Buraschi's parameters
lambda_vector= np.array([4.612, 2.891]) # Buraschi's parameters
#rho_bar= np.array([0.1453, -0.1968]) # SP-Appl paramters
#lambda_vector= np.array([-3.06536644, 6.83538405]) # SP-Appl paramters
r=0.02
# Generating prices for Wishart Process
ln_P = np.zeros((N, paths, 2))
ln_P[0,:]= np.array([1,1])
#W_P= ss.norm.rvs(loc=0, scale=1, size=(N - 1, paths, 2 ))
#generating W_P with estimated correlations
dB= ss.norm.rvs(loc=0, scale=1, size=(N - 1, paths, 2 ))
W_P= np.sqrt(1-rho_bar.T @ rho_bar) * dB + W[:,:,] @ rho_bar
for path in range(paths):
    for t in range(N-1):
        # Here wo do matrix calculations (sqrtm, multiplication) for np.array objects as matrices, a point to check whether it works correctly
        mu=r + Sigma_WP[t,path] @ lambda_vector
        drift= dt * (mu - 0.5 * np.diag(Sigma_WP[t,path]))
        # without - 0.5 * np.diag(Sigma_WP[t,path]) drift as in Da Fonseca model
        #drift= dt * (mu )
        diffusion= np.sqrt(dt) * np.matmul(np.real(scipy.linalg.sqrtm(Sigma_WP[t,path])) , W_P[t,path])
        #ln_P[t+1, path]= ln_P[t, path] + dt * (mu - 0.5 * np.diag(Sigma_WP[t,path])) + np.sqrt(dt) * np.matmul(scipy.linalg.sqrtm(Sigma_WP[t,path]) , W_P[t,path])
        ln_P[t+1, path]= ln_P[t, path] + drift + diffusion
P= np.exp(ln_P)

In [ ]:
# Computing returns
ret= np.zeros((paths, N-1, 2))
for path in range(paths):
  for t in range(N-1):
    ret[path,t]= ln_P[t+1,path]- ln_P[t,path]
cov= np.zeros((paths, 2, 2))

# Computing Covariances
for path in range(paths):
  cov[path]= np.cov(ret[path,:].T[0],ret[path,:].T[1] )
mean_cov= np.mean(cov, axis=0)

# Computing Ann. returns
Ann_ret= np.zeros((paths, 2))
for path in range(paths):
  Ann_ret[path]= ln_P[-1,path]- ln_P[0,path]
Ann_ret_mean= np.mean(Ann_ret,axis=0)

In [ ]:
#for Buraschi parameters:

# Theoretical Ann. return
np.array([[2.84669363],
        [3.0175595 ]])
# Theoretical Covariance
np.array([[0.3741894 , 0.38773162],
        [0.38773159, 0.42523051]])

# Simulated Ann. return
np.array([2.67252079, 2.82851982])

# Simulated Ann Covariance
np.array([[0.3709475, 0.385015 ],
       [0.385015 , 0.4232525]])
#np.array([[0.00148379, 0.00154006],
#       [0.00154006, 0.00169301]]) daily covariance

array([[0.00148379, 0.00154006],
       [0.00154006, 0.00169301]])

In [ ]:
#for SP-AAPL paramters

# Theoretical Ann return
np.array([0.1135, 0.4572])
# Theoretical Covariance
np.array([[0.0219, 0.0235],
       [0.0235, 0.0745]])

# Simulated Ann return
np.array([0.11977383, 0.47068118])

# Simulated Ann Covariance
np.array([[0.02443751, 0.02714514],
       [0.02714514, 0.08073289]])

#np.array([[9.77500547e-05, 1.08580549e-04],
#       [1.08580549e-04, 3.22931567e-04]]) daily covariance
#np.array([[0.0001, 0.0001],
#       [0.0001, 0.0003]]) daily covaraince


array([[0.0001, 0.0001],
       [0.0001, 0.0003]])

In [ ]:
250*np.array([[0.00148379, 0.00154006],
       [0.00154006, 0.00169301]])

array([[0.3709475, 0.385015 ],
       [0.385015 , 0.4232525]])

Plotting the allocations vs parameter c

In [ ]:
# Define time horizon range t for tests
time_horizon = np.linspace(0, 2, 100)
C=np.array([0.1, 0.5, 1, 2, 2.5])
# Compute portfolio allocations for both Theorems across the time horizon

fig = go.Figure()

for c in C:
  #Q_W_c, M_W_c= c* Q_W, c**2 * M_W
  beta_c= c * beta
  portfolio_allocations_M = []
  portfolio_allocations_W = []
  portfolio_allocations_SW = []
  portfolio_allocations_MMSW_S0 = []
  portfolio_allocations_MMSW_S1 = []
  for tau in time_horizon:
      # Compute the block matrix F_ij(t) = exp( t * A )
      #A = np.block([[Gamma, -2 * Lambda], [C, -Gamma.T]])
      #F = expm(t * A)

      # Extract F_11, F_12, F_21, F_22 from F
      #F_21 = F[2:, :2]
      #F_22 = F[2:, 2:]
      #beta_c= c * beta_T
      Sigma_inf_W_c= c * Sigma_inf_W
      lambda_vector_W_c= np.asmatrix(np.linalg.solve(Sigma_inf_W_c,mu_T-r)).T

      # Compute B(t)
      #B_t = np.linalg.inv(F_22) @ F_21
      # Merton Portfolio Strategy under Black Scholes Process
      portfolio_allocation_M_V= portfolio_allocation_M(lambda_vector_M)
      #portfolio_allocation_M_V= portfolio_allocation_M(lambda_vector_B)

      # Buratchi Portfolio Strategy under Wishart Process
      #portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_W, M_W_c,Q_W_c,rho_bar_W,tau)
      portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_W_c, M_W,Q_W,rho_bar_W,tau)
      #portfolio_allocation_W_V = portfolio_allocation_W(lambda_vector_B, M_B,Q_B,rho_bar_B,tau)

      # Theorem 1: Optimal Portfolio Strategy under Shifted Wishart Process
      #inv_Sigma_SW = np.linalg.inv(Sigma_SW)
      portfolio_allocation_SW_V = portfolio_allocation_SW(lambda_vector_SW, M_SW,Q_SW,rho_bar_SW,Sigma_inf_SW,sighat_SW,tau)
      #portfolio_allocation_SW_V = portfolio_allocation_SW(lambda_vector_B, M_BS,Q_BS,rho_bar_BS,Sigma_inf_BS,sighat_BS, tau)

      # Theorem 2: Optimal Portfolio Strategy under MMSW Process
      # Simulating a Markov chain that switches between states 0 and 1
      #state = np.random.choice([0, 1])  # Simulating the current state of the Markov chain
      #state = np.random.choice(2, p=transition_matrix[current_state])
      # The state-dependent widehat_Sigma
      #widehat_Sigma_MC = widehat_Sigma if state == 1 else np.zeros_like(widehat_Sigma)

      # Use the state-dependent widehat_Sigma_MC in the MMSW process
      #inv_Sigma_MW = np.linalg.inv(Sigma_MW)
      # MW portfolio for normal state
      portfolio_allocation_MMSW_V_S0 = portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, 0, tau)
      #portfolio_allocation_MMSW_V_S0 = portfolio_allocation_MMSW(lambda_vector_B, M_BM, Q_BM, rho_bar_BM, Sigma_inf_BM,sighat_BM, 0, tau)
      # MW portfolio for Shifted State
      portfolio_allocation_MMSW_V_S1 = portfolio_allocation_MMSW(lambda_vector_MW, M_MW, Q_MW, rho_bar_MW, Sigma_inf_MW,sighat_MW, 1, tau)
      #portfolio_allocation_MMSW_V_S1 = portfolio_allocation_MMSW(lambda_vector_B, M_BM, Q_BM, rho_bar_BM, Sigma_inf_BM,sighat_BM, 1, tau)

      # Store the results
      portfolio_allocations_M.append(np.array(portfolio_allocation_M_V))
      portfolio_allocations_W.append(np.array(portfolio_allocation_W_V))
      portfolio_allocations_SW.append(np.array(portfolio_allocation_SW_V))
      portfolio_allocations_MMSW_S0.append(np.array(portfolio_allocation_MMSW_V_S0))
      portfolio_allocations_MMSW_S1.append(np.array(portfolio_allocation_MMSW_V_S1))

  # Convert lists to numpy arrays for easier manipulation
  #portfolio_allocations_SW = np.array(portfolio_allocations_SW)
  #portfolio_allocations_MMSW_S0 = np.array(portfolio_allocations_MMSW_S0)
  #portfolio_allocations_MMSW_S1 = np.array(portfolio_allocations_MMSW_S1)

  # Create Plotly graphs to visualize the results

  # Add traces for both components of the portfolio allocation under Merton
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_M)[:, 0], mode='lines', name='M Portfolio Allocation 1'))
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_M)[:, 1], mode='lines', name='M Portfolio Allocation 2'))

  # Add traces for both components of the portfolio allocation under Wishart
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_W)[:, 0], mode='lines', name='W Portfolio Allocation 1 for c= '+ str(c)))
  fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_W)[:, 1], mode='lines', name='W Portfolio Allocation 2 for c= '+ str(c)))

  # Add traces for both components of the portfolio allocation under Shifted Wishart
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_SW)[:, 0], mode='lines', name='SW Portfolio Allocation 1'))
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_SW)[:, 1], mode='lines', name='SW Portfolio Allocation 2'))

  # Add traces for both components of the portfolio allocation under MMSW State 0
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 0], mode='lines', name='MMSW Normal State Portfolio Allocation 1', line=dict(dash='dash')))
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_MMSW_S0)[:, 1], mode='lines', name='MMSW Normal State Portfolio Allocation 2', line=dict(dash='dash')))

  # Add traces for both components of the portfolio allocation under MMSW State 1
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 0], mode='lines', name='MMSW Chrisis State Portfolio Allocation 1', line=dict(dash='dash')))
  #fig.add_trace(go.Scatter(x=time_horizon, y=np.squeeze(portfolio_allocations_MMSW_S1)[:, 1], mode='lines', name='MMSW Chrisis State Portfolio Allocation 2', line=dict(dash='dash')))

  # Customize the layout
  fig.update_layout(
      title="Portfolio Allocations under Wishart Process for different values of c",
      xaxis_title="Time Horizon (T - t)",
      yaxis_title="Portfolio Allocation",
      legend_title="Portfolio Components",
  )

  fig.write_html('portfolio_allocations_shifted_wishart_mmsw.html')

# Display the figure
fig.show()
